# SCOPE wavelet shrinkage — Experiments I, II and III (Python translation)

This notebook is a line-by-line translation of the canonical MATLAB program
**`SCOPE_Exp123_BN`** (file `SCOPE_Exp123_BN(1).m`, 2026-10-30) into Python
(NumPy / SciPy / pandas / Matplotlib). It reproduces the three simulation
experiments of the SCOPE paper and, at the end, **every figure**, with
adjustable fonts and LaTeX-typeset labels.

| Part | Content | MATLAB counterpart |
|---|---|---|
| 0 | Configuration (design, run mode, output folder, parallel jobs) | `SCOPE_Exp123_BN(mode, experiment, Ns, snrs, M)` |
| 1 | Wavelet matrices, filters, test signals, SNR scaling | `MakeONFilterExt`, `Wavmat`, `e1_make_DJ_signal`, `scope_scale_to_SNR` |
| 2 | Gaussian null-risk calibration of SCOPE scales | `scope_B_for_N`, `e1_nullrisk`, `e1_compute_constants`, `e2_*`, `calibrate_E3` |
| 3 | SCOPE / N-SCOPE rules and plug-in SURE | `e1_cdf_centered`, `e2_cdf_centered`, `neighborhoods_E3`, `apply_E3`, `select_E3` |
| 4 | **Experiment I** — fixed calibration, 6 generators × 4 radii | `e1_*` |
| 5 | **Experiment II** — SURE selection of $(k, r, b)$ | `e2_*` |
| 6 | **Experiment III** — 4 frozen SCOPE recipes vs. 20 comparators | `e3_*`, `E3_legacy_methods`, `E3_extended_methods`, … |
| 7 | Run the experiments | main loop |
| 8 | **All figures** from the saved CSV files | `e1_make_barplots`, `e2_condition_plots`, `plot_condition`, … |

### What is identical to MATLAB, and what is not
* **Identical:** design, filters and wavelet matrices, signals, SNR definition,
  rounded-$B_N$ calibration equations and tolerances, every estimator (formulas, grids,
  floors, tie-breaking order), seeds per condition (`seed + 100*is + iq`),
  output folder names, CSV file names and column names.
* **Not bit-identical:** the Gaussian noise. MATLAB's `randn` (Ziggurat on
  the Mersenne Twister) cannot be reproduced exactly with NumPy, so this
  notebook uses NumPy's `MT19937` generator with the same seed *numbers*.
  Results therefore agree with MATLAB **within Monte Carlo error**, not to
  the last digit. Paired comparisons (same noise for all methods within a
  replication) are preserved exactly.
* **Not needed:** MATLAB checkpoint migration (BAMS depth repair, BAMSO/CYV
  upgrade). Python checkpoints are new, and the methods are already the
  FINAL versions (#5 author BAMS with depth $L=J-3$, #23 Chang–Yu–Vetterli
  context). Functions that are present in the MATLAB file but never called
  (`scope_both`, `caucau_*`, `lnws_*`, `ncpmean2_*`, `sendur_selesnick_1d`,
  `my_periodic_dwt`, `benchmark_*`) are not translated.

### How to use
1. Set the configuration in **Part 0** (start with `MODE = "smoke"`).
2. Run all cells. The original Python/MATLAB result folders remain untouched: simulations write CSVs and checkpoints to
   `SCOPE_Exp123_results_BNrounded/`; an interrupted run resumes where it stopped.
3. Part 8 rebuilds all figures from the CSV files only, so fonts and labels
   can be changed and figures regenerated without re-running simulations.

## Part 0 — Configuration

* `MODE = "smoke"` runs the MATLAB smoke design ($N=512$, $M=2$, two
  signals, root SNR 5): a functionality check, **not** publishable evidence.
* `MODE = "full"` runs the complete design: $N\in\{512,1024,2048\}$, root
  SNR $\in\{3,5,7\}$, four Donoho–Johnstone signals, $M=1000$ paired
  replications per condition. Override `NS`, `SNRS`, `M` for partial runs,
  exactly like `SCOPE_Exp123_BN('full','Exp3',1024,[3 5 7],1000)`.
* Supported rounded budgets: `N=128: B_N=1.75`, `512:3.5`, `1024:5`,
  `2048:7.5`, `4096:11.5` (for `4096`, `j0=4` instead of `3`).
* `N_JOBS` runs the signal × SNR conditions of one experiment in parallel
  (each condition has its own seed and checkpoint, so results do not depend
  on `N_JOBS`). Use `N_JOBS = 1` for sequential runs.

In [ ]:
MODE = "smoke"              # "smoke" or "full"
EXPERIMENTS = ["Exp1", "Exp2", "Exp3"]   # any subset of ["Exp1","Exp2","Exp3"]
NS = None                   # None -> [512, 1024, 2048] in full mode; e.g. [1024]
SNRS = None                 # None -> [3, 5, 7] in full mode;  e.g. [5]
M = None                    # None -> 1000 in full mode
OUT_BASE = "SCOPE_Exp123_results_BNrounded"   # results folder (created next to the notebook)
N_JOBS = 1                  # parallel conditions (joblib); -1 = all cores
RESUME = True               # reuse finished / partial checkpoints
RUN_SIMULATIONS = True      # False -> skip Part 7 and only draw figures (Part 8)

### Imports and small utilities
Only NumPy, SciPy, pandas and Matplotlib are required (`joblib` is optional,
for `N_JOBS != 1`). Floating-point warnings from deliberately guarded
operations (e.g. `exp` overflow inside `np.where`) are silenced, exactly as
MATLAB silently returns `Inf`/`NaN` there before the guard picks a branch.

In [ ]:
import os, time, json, pickle, math, warnings, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy import integrate, optimize
from scipy.special import erf, erfc, erfcinv, erfinv, gammaln

np.seterr(all="ignore")
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=integrate.IntegrationWarning)

REALMIN = np.finfo(float).tiny      # MATLAB realmin
EPS = np.finfo(float).eps           # MATLAB eps
MAD_CONST = 0.6744897501960817      # Phi^{-1}(3/4)

def log(msg):
    """Timestamped progress message (MATLAB fprintf + diary)."""
    print(time.strftime("%H:%M:%S"), msg, flush=True)

def resolve_design(mode, Ns, snrs, M):
    """Mirror the argument handling at the top of SCOPE_Exp123_BN."""
    assert mode in ("smoke", "full"), "Mode must be full/smoke"
    if mode == "smoke":
        Ns, M, snrs = [512], 2, [5]
    else:
        Ns = [512, 1024, 2048] if Ns is None else list(Ns)
        snrs = [3, 5, 7] if snrs is None else list(snrs)
        M = 1000 if M is None else int(M)
    Ns = list(dict.fromkeys(int(n) for n in Ns))          # unique, stable
    snrs = list(dict.fromkeys(float(s) for s in snrs))
    assert all(n >= 16 and (n & (n - 1)) == 0 for n in Ns), "Ns must be powers of two >= 16."
    assert all(np.isfinite(s) and s > 0 for s in snrs), "SNRs must be positive finite numbers."
    assert M >= 1
    # Separate result folders for non-default SNR sets (MATLAB snrTag).
    snr_tag = "" if snrs == [3.0, 5.0, 7.0] else "_SNR" + "_".join(
        f"{s:.8g}".replace(".", "p") for s in snrs)
    return Ns, [int(s) if float(s).is_integer() else s for s in snrs], M, snr_tag

def fmt_snr(s):
    """MATLAB sprintf('%d', snr) for file names (integers print without .0)."""
    return f"{int(s)}" if float(s).is_integer() else f"{s:g}"

def save_pickle(obj, path):
    tmp = str(path) + ".tmp"
    with open(tmp, "wb") as fh:
        pickle.dump(obj, fh, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)                      # atomic: never a half-written checkpoint

def load_pickle(path):
    with open(path, "rb") as fh:
        return pickle.load(fh)

def make_rng(seed):
    """Mersenne Twister generator seeded like MATLAB rng(seed,'twister').
    NOTE: the normal deviates differ from MATLAB's randn (see introduction)."""
    return np.random.Generator(np.random.MT19937(int(seed)))

def run_conditions(func, jobs, n_jobs):
    """Run independent signal x SNR conditions, optionally in parallel."""
    if n_jobs == 1 or len(jobs) <= 1:
        return [func(*j) for j in jobs]
    from joblib import Parallel, delayed
    return Parallel(n_jobs=n_jobs, backend="loky")(delayed(func)(*j) for j in jobs)

## Part 1 — Wavelets, filters and test signals

### Orthonormal filters (`MakeONFilterExt`)
The exact tap values of the supplied `MakeONFilterExt.m` are used (no
toolbox): **Symmlet-4** (8 taps) for Doppler and HeaviSine, **Daubechies-3**
(6 taps) for Bumps, **Haar** for Blocks. Each filter is normalised to unit
$\ell_2$ norm.

In [ ]:
def make_on_filter(ftype, par):
    t = ftype.lower()
    if t == "haar":
        h = np.array([1.0, 1.0]) / np.sqrt(2)
    elif t == "daubechies":
        assert par == 6, "Use six-tap Daubechies-3 for Bumps"
        h = np.array([0.332670552950082615998512, 0.806891509311092576494494,
                      0.459877502118491570095152, -0.135011020010254588696390,
                      -0.085441273882026661692819, 0.0352262918857095366027407])
    elif t == "symmlet":
        assert par == 4, "Use Symmlet-4 for Doppler/HeaviSine"
        h = np.array([-0.07576571478950221, -0.029635527646002493,
                      0.497618667632775, 0.8037387518051321,
                      0.29785779560530606, -0.09921954357663353,
                      -0.012603967262031304, 0.032223100604051466])
    else:
        raise ValueError(f"Unrecognized filter {ftype}")
    return h / np.linalg.norm(h)

### Periodic orthonormal DWT as a sparse matrix (`Wavmat`)
`Wavmat(h, N, k0, shift)` builds the $N\times N$ orthogonal matrix $W$ of a
periodic DWT with $k_0$ decomposition steps. With $k_0 = J - j_0$ and
$j_0 = 3$ for $N=128,512,1024,2048$ (or $j_0=4$ for $N=4096$), $d = Wy$ is ordered as
$$d = \big[\,\underbrace{c_{j_0}}_{2^{j_0}\ \text{scaling}},\ d_{j_0},\ d_{j_0+1},\ \dots,\ d_{J-1}\,\big],
\qquad |d_j| = 2^j,$$
so level $j$ occupies (1-based) indices $2^j+1,\dots,2^{j+1}$, i.e. the
Python slice `[2**j : 2**(j+1)]`. The high-pass filter is
$g_i = (-1)^i h_{L+1-i}$ (MATLAB: `fliplr(conj(h).*(-1).^(1:L))`), and the
periodic phase convention `shift = 2` of the supplied code is kept.

Because $W$ is orthogonal, the mean squared error can be computed directly
in the wavelet domain (Parseval):
$\tfrac1N\|\hat f - f\|^2 = \tfrac1N\|\hat\theta - \theta\|^2$ with $\theta = Wf$.

In [ ]:
def wavmat(h, N, k0, shift=2):
    J = int(round(np.log2(N)))
    assert 2 ** J == N and 1 <= k0 <= J
    h = np.asarray(h, float).ravel()
    nh = h.size
    g = np.flip(h * (-1.0) ** np.arange(1, nh + 1))
    old = sp.identity(2 ** (J - k0), format="csr")
    for k in range(k0, 0, -1):
        nout = 2 ** (J - k); nin = 2 * nout
        rows = np.tile(np.arange(1, nout + 1), nh)                 # repmat((1:nout)',nh,1)
        taps = np.repeat(np.arange(1, nh + 1), nout)               # kron((1:nh)',ones(nout,1))
        cols = np.mod(taps + 2 * rows - shift - 1, nin)            # 0-based column
        H = sp.csr_matrix((np.repeat(h, nout), (rows - 1, cols)), shape=(nout, nin))
        G = sp.csr_matrix((np.repeat(g, nout), (rows - 1, cols)), shape=(nout, nin))
        old = sp.vstack([old @ H, G]).tocsr()
    return old

FILTERS = {1: ("Symmlet", 4, "Symmlet4_8tap"),
           2: ("Daubechies", 6, "Daubechies3_6tap"),
           3: ("Haar", 0, "Haar_2tap")}

def wavelet_bank(N, j0=3, shift=2, check_vec_freq=1.2345):
    """W matrices and low-pass filters for whichW = 1 (Symmlet), 2 (Daub), 3 (Haar),
    each checked for orthogonality as in e1_wavelet_matrix."""
    J = int(round(np.log2(N)))
    bank, hbank = {}, {}
    v = np.sin(np.arange(1, N + 1) * check_vec_freq)
    for k, (name, par, _) in FILTERS.items():
        h = make_on_filter(name, par)
        W = wavmat(h, N, J - j0, shift)
        err = np.linalg.norm(W.T @ (W @ v) - v) / np.linalg.norm(v)
        assert err < 3e-8, f"Wavmat failed the orthogonality check for {name}"
        bank[k], hbank[k] = W, h
    return bank, hbank

### Donoho–Johnstone test signals and root-SNR scaling
The four classical signals are sampled at $t_i = i/N$, $i=1,\dots,N$.
**Root SNR** is the ratio of standard deviations,
$$\mathrm{SNR} = \frac{\operatorname{SD}(f)}{\sigma}, \qquad \operatorname{SD}(f)=\Big(\tfrac1N\sum_i (f_i-\bar f)^2\Big)^{1/2},\quad \sigma = 1,$$
**not** the power ratio. Each signal is rescaled so that
$\operatorname{SD}(f) = \mathrm{SNR}$ (population SD, divisor $N$); the noise
$\varepsilon_i \stackrel{iid}{\sim} N(0,1)$ is never standardised.

In [ ]:
DJ_POS = np.array([0.10, 0.13, 0.15, 0.23, 0.25, 0.40, 0.44, 0.65, 0.76, 0.78, 0.81])
DJ_HTB = np.array([4.0, 5.0, 3.0, 4.0, 5.0, 4.2, 2.1, 4.3, 3.1, 5.1, 4.2])
DJ_HTK = np.array([4.0, -5.0, 3.0, -4.0, 5.0, -4.2, 2.1, 4.3, -3.1, 2.1, -4.2])
DJ_W = np.array([0.005, 0.005, 0.006, 0.010, 0.010, 0.030, 0.010, 0.010, 0.005, 0.008, 0.005])
SIGNALS = ["Doppler", "HeaviSine", "Bumps", "Blocks"]

def make_dj_signal(name, N):
    """Returns (f0, filter label, whichW) exactly as e1_make_DJ_signal."""
    t = np.arange(1, N + 1) / N
    n = name.lower()
    if n == "doppler":
        f0 = np.sqrt(t * (1 - t)) * np.sin(2 * np.pi * 1.05 / (t + 0.05)); w = 1
    elif n == "heavisine":
        f0 = 4 * np.sin(4 * np.pi * t) - np.sign(t - 0.30) - np.sign(0.72 - t); w = 1
    elif n == "bumps":
        f0 = np.zeros(N)
        for p, hgt, wd in zip(DJ_POS, DJ_HTB, DJ_W):
            f0 += hgt / (1 + np.abs((t - p) / wd)) ** 4
        w = 2
    elif n == "blocks":
        f0 = np.zeros(N)
        for p, hgt in zip(DJ_POS, DJ_HTK):
            f0 += hgt * (t >= p)
        w = 3
    else:
        raise ValueError(f"Unknown signal: {name}")
    assert np.all(np.isfinite(f0)) and np.mean(f0 ** 2) > 0
    return f0, FILTERS[w][2], w

def scale_to_snr(f0, snr):
    """f = f0 * SNR / SD(f0), population SD (MATLAB std(f,1))."""
    f0 = np.asarray(f0, float).ravel()
    s = np.std(f0)                     # ddof=0  == std(f0,1)
    assert np.isfinite(snr) and snr > 0 and s > 0
    f = f0 * (snr / s)
    assert abs(np.std(f) - snr) < 1e-9 * max(1, snr)
    return f

## Part 2 — Gaussian null-risk calibration

### Centred generators $F^*$
SCOPE attenuates a coefficient $d$ by $F^*(\lambda\, s)^k$, where $s$ is the
local RMS of $d$ (Part 3) and $F^*(u) = 2F(u) - 1$ is a centred CDF on
$u\ge0$. The six generators, with the **paper's parameter conventions**, and
their derivatives (needed by SURE) are

| Generator | $F^*(u)$ | $F^{*\prime}(u)$ |
|---|---|---|
| Normal | $\operatorname{erf}(u/\sqrt2)$ | $\sqrt{2/\pi}\,e^{-u^2/2}$ |
| Logistic | $\tanh u$ | $1-\tanh^2u$ |
| Laplace | $1-e^{-u}$ | $e^{-u}$ |
| Sech | $\tfrac2\pi\arctan(\sinh(\pi u/2))$ | $1/\cosh(\pi u/2)$ |
| Uniform | $\min(u,1)$ | $\mathbb 1\{u<1\}$ |
| Cauchy | $\tfrac2\pi\arctan u$ | $\tfrac2\pi(1+u^2)^{-1}$ |

In [ ]:
GENERATORS = ["Normal", "Logistic", "Laplace", "Sech", "Uniform", "Cauchy"]

def cdf_centered(F, u, deriv=False):
    """F*(u) for u>=0 (negative u clipped to 0), optionally with F*'(u).
    Same formulas as e1_cdf_centered / e2_cdf_centered / fstar_E3."""
    u = np.maximum(0.0, u)
    f = F.lower()
    if f == "normal":
        g = erf(u / np.sqrt(2)); gp = np.sqrt(2 / np.pi) * np.exp(-u ** 2 / 2) if deriv else None
    elif f == "logistic":
        g = np.tanh(u); gp = 1 - g ** 2 if deriv else None
    elif f == "laplace":
        g = -np.expm1(-u); gp = np.exp(-u) if deriv else None
    elif f == "sech":
        g = (2 / np.pi) * np.arctan(np.sinh(np.pi * u / 2)); gp = 1 / np.cosh(np.pi * u / 2) if deriv else None
    elif f == "uniform":
        g = np.minimum(u, 1.0); gp = (u < 1).astype(float) if deriv else None
    elif f == "cauchy":
        g = (2 / np.pi) * np.arctan(u); gp = (2 / np.pi) / (1 + u ** 2) if deriv else None
    else:
        raise ValueError(f"Unrecognized centered CDF: {F}")
    g = np.clip(g, 0.0, 1.0)
    return (g, gp) if deriv else g

### Section 4 rounded $B_N$ calibration and level schedule
Let $N=2^J$. The fixed, **prespecified** null-noise budget $B_N$ is
$1.75$ for $N=128$, $3.5$ for $512$, $5$ for $1024$, $7.5$ for $2048$,
and $11.5$ for $4096$. The first shrunk wavelet level is $j_0=3$ for
$N\le2048$ and $j_0=4$ for $N=4096$. These are the values in the
October 30 MATLAB source; the previous data-independent but **unrounded**
$\eta(N)$ half-attenuation calibration has been replaced.

Under pure Gaussian noise ($\theta=0$, $\sigma=1$), the null risk of a
local-radius-$r$ SCOPE rule is
$$R_{F,r,k}(a) = \mathbb{E}\!\left[
  \frac{U}{m}\,F^*\!\left(a\sqrt{U/m}\right)^{2k}\right],
  \qquad U\sim\chi^2_m,\quad m=2r+1.$$
The calibration is **directly**
$$R_{F,r,k}(a_{F,r,k,j})=\frac{B_N}{2^j},
\qquad j=j_0,\ldots,J-1,$$
so the expected retained null-noise energy in each detail level is $B_N$.
For a noise estimate $\hat\sigma$ and SURE-grid multiplier $b$ (fixed
$b=1$ in Experiment I), the estimator uses
$\lambda_j=b\,a_{F,r,k,j}/\hat\sigma$.
No signal truth, oracle optimization, or SURE enters the calibration.


In [ ]:
# Exact Section 4 table, shared by Experiments I, II and III.
ROUNDED_BN = {128: (1.75, 3), 512: (3.5, 3), 1024: (5.0, 3),
              2048: (7.5, 3), 4096: (11.5, 4)}

def scope_B_for_N(N, verbose=True):
    """Return (B_N,j0) from Section 4 in SCOPE_Exp123_BN(1).m (Oct. 30).
    The five allowed N are deliberate; do not interpolate the budgets."""
    if N not in ROUNDED_BN:
        raise ValueError(f"Rounded Section 4 B_N is specified only for N = "
                         f"{tuple(ROUNDED_BN)} (received N={N}).")
    B_N, j0 = ROUNDED_BN[N]
    J = int(round(np.log2(N)))
    assert B_N > 0 and B_N < 2 ** j0 and 2 * max(range(4)) + 1 <= 2 ** j0
    assert J > j0
    if verbose:
        log(f"Section 4 calibration: N={N}, B_N={B_N:g}, j0={j0}, "
            f"finest target={B_N/(N/2):.10g}")
    return B_N, j0

def null_risk(a, F, r, k):
    """R_{F,r,k}(a) = E[(U/m) F*(a sqrt(U/m))^(2k)], U ~ chi2_m (e1_nullrisk / e2_nullrisk)."""
    if a <= 0:
        return 0.0
    m = 2 * r + 1; nu = m / 2
    logC = -nu * np.log(2) - gammaln(nu) - np.log(m)
    fun = lambda u: np.exp(nu * np.log(max(u, REALMIN)) - 0.5 * u + logC) * cdf_centered(F, a * np.sqrt(u / m)) ** (2 * k)
    val = integrate.quad(fun, 0, np.inf, epsabs=2e-10, epsrel=1e-8, limit=200)[0]
    return float(min(1.0, max(0.0, val)))

def solve_scale(F, r, k, target):
    """Bracket by doubling, then solve R_{F,r,k}(a) = B_N/2^j."""
    assert 0 < target < 1, f"Invalid per-coefficient null risk target: {target}"
    fun = lambda z: null_risk(z, F, r, k)
    upper = 1.0
    while fun(upper) < target:
        upper *= 2
        assert upper < 1e6, "Failed to bracket null-risk equation."
    return optimize.brentq(lambda z: fun(z) - target, 0.0, upper,
                           xtol=1e-10, rtol=4 * EPS, maxiter=500)

def compute_constants(N, generators, radii, kgrid, j0=None, B_N=None, cache=None):
    """Calibrated a[iF,ik,ir,jj], solving R(a)=B_N/2**j for every level.
    The persisted calibration is tagged with the B_N table and source version;
    no old eta-based constants can be reused."""
    J = int(round(np.log2(N)))
    B_fixed, j0_fixed = scope_B_for_N(N, verbose=False)
    j0 = j0_fixed if j0 is None else j0
    B_N = B_fixed if B_N is None else B_N
    assert j0 == j0_fixed and B_N == B_fixed, "This notebook uses fixed Section 4 rounded B_N only."
    meta = dict(rule="Section4_rounded_BN_over_2powj_20261030", N=N, J=J,
                j0=j0, B_N=B_N, generators=list(generators),
                radii=list(radii), kgrid=list(kgrid))
    if cache is not None and Path(cache).exists():
        obj = load_pickle(cache)
        assert obj["meta"] == meta, f"Cached calibration belongs to different settings: {cache}"
        a = obj["a"]
    else:
        a = np.zeros((len(generators), len(kgrid), len(radii), J - j0))
        for iF, F in enumerate(generators):
            for ik, k in enumerate(kgrid):
                for ir, r in enumerate(radii):
                    for jj in range(J - j0):
                        j = j0 + jj
                        a[iF, ik, ir, jj] = solve_scale(F, r, k, B_N / 2 ** j)
            log(f"  calibrated {F:9s} k={list(kgrid)}, r={list(radii)} (B_N={B_N:g})")
        if cache is not None:
            save_pickle(dict(meta=meta, a=a), cache)
    assert a.shape == (len(generators), len(kgrid), len(radii), J - j0)
    return a

def check_constants(a, generators, radii, kgrid, N, B_N):
    """Match MATLAB e1_check_constants / e2_check_k5_constants at finest level."""
    ik = list(kgrid).index(5)
    target = B_N / (N // 2)
    for iF, F in enumerate(generators):
        for ir, r in enumerate(radii):
            got = null_risk(a[iF, ik, ir, -1], F, r, 5)
            assert abs(got - target) < 1e-5, \
                f"Null-risk calibration root failed: N={N}, F={F}, r={r}, got={got}, target={target}"

def constants_table(a, generators, radii, kgrid, N, j0, B_N, with_k=True):
    """Match MATLAB Experiment1_Constants / Experiment2_ScaleConstants CSV schemas."""
    J = int(round(np.log2(N)))
    rows = []
    for iF, F in enumerate(generators):
        for ik, k in enumerate(kgrid):
            for ir, r in enumerate(radii):
                for jj in range(J - j0):
                    j = j0 + jj
                    row = {"Generator": F}
                    if with_k:
                        row["K"] = k
                    row.update({"Radius": r, "Level_j": j, "Budget_BN": B_N,
                                "TargetNullRisk": B_N / 2 ** j,
                                "a_NullRisk": a[iF, ik, ir, jj]})
                    rows.append(row)
    return pd.DataFrame(rows)


## Part 3 — SCOPE / N-SCOPE rules and plug-in SURE

For detail level $j$ with coefficients $d_{j,l}$ and radius $r$, the
**periodic local RMS** is
$$S_{j,l}(r) = \Big(\tfrac{1}{2r+1}\textstyle\sum_{|t|\le r} d_{j,l+t}^2\Big)^{1/2}\quad(\text{indices mod } 2^j),$$
so $S(0)=|d|$ (SCOPE) and $r\ge1$ gives N-SCOPE. The estimator is
$$\hat\theta_{j,l} = d_{j,l}\,w_{j,l},\qquad w_{j,l} = F^*\!\big(\lambda_j S_{j,l}(r)\big)^{k},\qquad \lambda_j = b\,a_{F,r,k,j}/\hat\sigma .$$

**Plug-in SURE** (Stein's unbiased risk estimate with $\hat\sigma$ held
fixed). Only the diagonal Jacobian enters; for the central coefficient,
$\partial S_{j,l}/\partial d_{j,l} = d_{j,l}/(m S_{j,l})$ with $m = 2r+1$, so
$$\frac{\partial\hat\theta_{j,l}}{\partial d_{j,l}} = w_{j,l} + \frac{k\,\lambda_j\, d_{j,l}^2}{m\,S_{j,l}}\,F^*(\lambda_jS_{j,l})^{k-1}F^{*\prime}(\lambda_jS_{j,l}),$$
$$\mathrm{SURE} = \sum_{j,l}\Big[d_{j,l}^2(w_{j,l}-1)^2 + 2\hat\sigma^2\,\partial_{d}\hat\theta_{j,l} - \hat\sigma^2\Big].$$
$(k,r,b)$ are chosen **globally** (summed over all levels $j_0,\dots,J-1$).
The noise level is the MAD of the finest level,
$\hat\sigma = \operatorname{median}|d_{J-1,\cdot}|/0.6744897501960817$.
The $2^{j_0}=8$ scaling coefficients are never shrunk.

In [ ]:
def level_slices(N, j0=3):
    """[(j, slice)] for detail levels j0..J-1 in Wavmat ordering."""
    J = int(round(np.log2(N)))
    return [(j, slice(2 ** j, 2 ** (j + 1))) for j in range(j0, J)]

def sigma_mad(D):
    """MAD noise estimate from the finest level; D is (N,) or (N, nb)."""
    N = D.shape[0]
    return np.median(np.abs(D[N // 2:]), axis=0) / MAD_CONST

def local_rms(dj, r):
    """S_{j,l}(r) along axis 0 (periodic). Works for (n,) and (n, nb) arrays."""
    if r == 0:
        return np.abs(dj)
    d2 = dj ** 2
    e = d2.copy()
    for lag in range(1, r + 1):
        e += np.roll(d2, lag, axis=0) + np.roll(d2, -lag, axis=0)
    return np.sqrt(e / (2 * r + 1))

def validate_sure_divergence():
    """e2_validate_divergence: finite-difference check of the diagonal derivative
    for all six generators, k in {3,5,7}, r in {0,1,3}."""
    d = np.array([0.24, -0.74, 0.51, 1.11, -0.42, 0.66, -0.35, 0.83])
    center, lam, h = 3, 0.57, 1e-6                     # MATLAB center=4 (1-based)
    for F in GENERATORS:
        for rr in (0, 1, 3):
            mm = 2 * rr + 1
            ids = np.mod(np.arange(center - rr, center + rr + 1), d.size)
            S = np.sqrt(np.sum(d[ids] ** 2) / mm)
            g, gp = cdf_centered(F, lam * S, deriv=True)
            for k in (3, 5, 7):
                analytic = g ** k + (k * lam * d[center] ** 2 / (mm * S)) * g ** (k - 1) * gp
                plus = d.copy(); plus[center] += h
                minus = d.copy(); minus[center] -= h
                sp_ = np.sqrt(np.sum(plus[ids] ** 2) / mm); sm_ = np.sqrt(np.sum(minus[ids] ** 2) / mm)
                fp = plus[center] * cdf_centered(F, lam * sp_) ** k
                fm = minus[center] * cdf_centered(F, lam * sm_) ** k
                assert abs((fp - fm) / (2 * h) - analytic) < 5e-6, f"SURE derivative check failed: {F}, k={k}, r={rr}"
    log("Passed finite-difference checks of the N-SCOPE diagonal divergence.")

## Part 4 — Experiment I: fixed calibration and neighbourhood radius

**Design.** For each signal and root SNR, $M$ paired replications
$y = f + \varepsilon$, $\varepsilon\sim N(0, I_N)$, are transformed once,
$d = Wy$, and **all 26 procedures see the same noise**:

* 24 fixed rules: 6 generators $F$ × radii $r\in\{0,1,2,3\}$, with $k=5$,
  $b=1$ and scales $\lambda_j = a_{F,r,5,j}/\hat\sigma$ (no SURE, no oracle);
  $r=0$ is SCOPE, $r\ge1$ is N-SCOPE.
* 2 references: universal hard and soft thresholding at
  $\hat\sigma\sqrt{2\log N}$ on all detail levels.

**Criteria.** Replication MSE $=\tfrac1N\sum_i(\hat f_i-f_i)^2$ (computed in
the wavelet domain), AMSE = mean over $M$ replications, RMSE = $\sqrt{\text{AMSE}}$
(not the mean of square roots), Monte Carlo SE = SD/$\sqrt M$, and paired
differences $r=0$ minus $r\in\{1,2,3\}$ with 95% normal intervals.

Replications are processed in batches of 100 (MATLAB `cfg.batchSize`); a
partial checkpoint after every batch makes a condition resumable.

In [ ]:
def e1_method_names(generators, radii):
    names = [f"{F}_r{r}" for F in generators for r in radii]
    return names + ["UniversalHard", "UniversalSoft"]

def e1_simulate_condition(f, W, a, cfg, names, tag, snr, filter_tag, partial_fn, seed):
    """Vectorised Monte Carlo of all 26 Experiment I rules for one signal/SNR."""
    N, Mrep, j0, J = cfg["N"], cfg["M"], cfg["j0"], cfg["J"]
    nmeth = len(names)
    trueD = W @ f
    lv = level_slices(N, j0)
    if cfg["resume"] and Path(partial_fn).exists():
        st = load_pickle(partial_fn)
        assert (st["N"] == N and st["M"] == Mrep and st["signal"] == tag and
                st["snr"] == snr and st.get("calibration_BN") == cfg["B_N"] and
                st.get("j0") == j0 and st["mse"].shape == (Mrep, nmeth)), \
            f"Old / incompatible E1 partial checkpoint: {partial_fn}"
        mse, sig_all, rep_first, completed = st["mse"], st["sigmahats"], st["repFirst"], st["completed"]
        rng = make_rng(0); rng.bit_generator.state = st["rngState"]
        log(f"  E1 resumed partial {tag} SNR {snr} at {completed}/{Mrep}")
    else:
        rng = make_rng(seed)
        mse = np.zeros((Mrep, nmeth)); sig_all = np.zeros(Mrep); rep_first = None; completed = 0
    for istart in range(completed, Mrep, cfg["batchSize"]):
        iend = min(Mrep, istart + cfg["batchSize"]); nb = iend - istart
        Y = f[:, None] + rng.standard_normal((N, nb))
        D = W @ Y                                         # one transform for all methods
        sig = sigma_mad(D)
        assert np.all(sig > 0) and np.all(np.isfinite(sig))
        sig_all[istart:iend] = sig
        if istart == 0:
            rep_first = dict(y=Y[:, 0].copy(), D=D[:, 0].copy(), sigmahat=sig[0])
        # Unshrunk scaling coefficients contribute equally to every method.
        err_scaling = np.sum((D[:2 ** j0] - trueD[:2 ** j0, None]) ** 2, axis=0)
        loss = np.tile(err_scaling, (nmeth, 1))           # [method x replication]
        thr = sig * np.sqrt(2 * np.log(N))
        for jj, (j, sl) in enumerate(lv):
            dj = D[sl]; target = trueD[sl][:, None]
            rms = [local_rms(dj, r) for r in cfg["radii"]]
            for iF, F in enumerate(cfg["generators"]):
                for ir in range(len(cfg["radii"])):
                    im = iF * len(cfg["radii"]) + ir
                    lam = a[iF, 0, ir, jj] / sig          # one lambda per replication
                    theta = dj * cdf_centered(F, rms[ir] * lam) ** cfg["k"]
                    loss[im] += np.sum((theta - target) ** 2, axis=0)
            hard = dj * (np.abs(dj) > thr)
            soft = np.sign(dj) * np.maximum(np.abs(dj) - thr, 0)
            loss[-2] += np.sum((hard - target) ** 2, axis=0)
            loss[-1] += np.sum((soft - target) ** 2, axis=0)
        mse[istart:iend] = loss.T / N
        save_pickle(dict(N=N, M=Mrep, signal=tag, snr=snr, calibration_BN=cfg["B_N"],
                         j0=j0, mse=mse, sigmahats=sig_all,
                         repFirst=rep_first, completed=iend,
                         rngState=rng.bit_generator.state), partial_fn)
        log(f"  E1 {tag} / SNR {snr}: {iend:4d}/{Mrep:4d} ({100 * iend / Mrep:5.1f}%)")
    return dict(signal=tag, snr=snr, filter=filter_tag, M=Mrep, names=names, mse=mse,
                sigmahat=sig_all, f=f, firstReplication=rep_first, waveletShift=cfg["shift"],
                snrDefinition=cfg["snrDefinition"], calibration_BN=cfg["B_N"], j0=j0)

def write_mse_csv(R, path, sigma_key="sigmahat"):
    """MSE_replications_*.csv: Replication, SigmaHat, one column per method."""
    T = pd.DataFrame({"Replication": np.arange(1, R["M"] + 1), "SigmaHat": R[sigma_key]})
    for i, nm in enumerate(R["names"]):
        T[nm] = R["mse"][:, i]
    T.to_csv(path, index=False)

def e1_compile_tables(results, cfg, names):
    """Experiment1_AMSE_all / PairedGains / AggregateRanking (e1_compile_tables)."""
    rows, gains, scores = [], [], []
    nR = len(cfg["radii"])
    for r in results:
        m = r["mse"].mean(axis=0); sd = r["mse"].std(axis=0, ddof=1); Mrep = r["M"]
        scores.append(m / m[0])                       # shared Normal-r0 reference
        for i, nm in enumerate(names):
            if i < len(cfg["generators"]) * nR:
                fi, rad = divmod(i, nR)
                family = cfg["generators"][fi]
                method = "SCOPE_Fixed" if rad == 0 else "NSCOPE_Fixed"
                base = r["mse"][:, fi * nR].mean()
                gain = 100 * (base - m[i]) / base
            else:
                fi, rad, family, method, gain = -1, -1, "Universal", "Reference", np.nan
            rows.append([r["signal"], r["snr"], r["filter"], nm, method, family, rad, Mrep,
                         m[i], sd[i], sd[i] / np.sqrt(Mrep), gain])
            if fi >= 0 and rad > 0:
                u = r["mse"][:, fi * nR] - r["mse"][:, i]
                du, se = u.mean(), u.std(ddof=1) / np.sqrt(Mrep)
                gains.append([r["signal"], r["snr"], family, rad, du, se, du - 1.96 * se, du + 1.96 * se, gain])
    overall = pd.DataFrame(rows, columns=["Signal", "SNR", "Filter", "Procedure", "Class", "Generator",
                                          "Radius", "M", "AMSE", "SD_MSE", "SE_AMSE", "GainPct_vs_sameF_r0"])
    gains = pd.DataFrame(gains, columns=["Signal", "SNR", "Generator", "Radius", "PairedDeltaAMSE",
                                         "PairedSE_Delta", "CI95_Lower", "CI95_Upper", "GainPercent"])
    rel = np.mean(scores, axis=0); order = np.argsort(rel, kind="stable")
    ranking = pd.DataFrame({"Procedure": np.array(names)[order], "MeanRelativeAMSE_to_Normal_r0": rel[order]})
    return overall, gains, ranking

def e1_condition_job(cfg, a, is_, iq, sig_name, snr, out_dir):
    """One signal/SNR condition of Experiment I (resumable)."""
    f0, filt, which = make_dj_signal(sig_name, cfg["N"])
    W = wavelet_bank(cfg["N"], cfg["j0"], cfg["shift"])[0][which]
    names = e1_method_names(cfg["generators"], cfg["radii"])
    cp = out_dir / f"checkpoint_{sig_name}_SNR{fmt_snr(snr)}.pkl"
    if cfg["resume"] and cp.exists():
        R = load_pickle(cp)
        assert (R["snrDefinition"] == cfg["snrDefinition"] and
                R["mse"].shape == (cfg["M"], len(names)) and
                R.get("calibration_BN") == cfg["B_N"] and R.get("j0") == cfg["j0"]), \
            f"Checkpoint incompatible with rounded B_N: {cp}"
        log(f"RESUMED E1 {sig_name} / SNR={snr} from checkpoint.")
    else:
        f = scale_to_snr(f0, snr)
        partial = out_dir / f"PARTIAL_{sig_name}_SNR{fmt_snr(snr)}.pkl"
        R = e1_simulate_condition(f, W, a, cfg, names, sig_name, snr, filt, partial,
                                  cfg["baseSeed"] + 100 * is_ + iq)
        save_pickle(R, cp)
        if partial.exists(): partial.unlink()
    write_mse_csv(R, out_dir / f"MSE_replications_{sig_name}_SNR{fmt_snr(snr)}.csv")
    return R

def run_experiment1(mode, N, Mrep, snrs, out_base, snr_tag, n_jobs=1, resume=True):
    smoke = mode == "smoke"
    B_N, j0 = scope_B_for_N(N)
    cfg = dict(N=N, J=int(round(np.log2(N))), j0=j0, B_N=B_N, k=5, radii=[0, 1, 2, 3],
               generators=list(GENERATORS), signals=list(SIGNALS), snrs=list(snrs),
               snrDefinition="std_signal_over_std_noise_iidN01", M=Mrep, batchSize=100,
               baseSeed=20261026 + N, resume=resume, shift=2)
    if smoke:
        cfg.update(signals=["Doppler", "Blocks"], snrs=[5], batchSize=min(6, Mrep))
        out_dir = Path(out_base) / f"Exp1_N{N}_SMOKE"
    else:
        out_dir = Path(out_base) / f"Exp1_N{N}_M{Mrep}{snr_tag}"
    out_dir.mkdir(parents=True, exist_ok=True)
    names = e1_method_names(cfg["generators"], cfg["radii"]); assert len(names) == 26
    log(f"Experiment I, N={N}: calibrating 6 CDF x 4 radii x {cfg['J'] - cfg['j0']} levels ...")
    a = compute_constants(N, cfg["generators"], cfg["radii"], [5], cfg["j0"], cfg["B_N"],
                          cache=out_dir / "calibration_E1.pkl")
    check_constants(a, cfg["generators"], cfg["radii"], [5], N, cfg["B_N"])
    constants_table(a, cfg["generators"], cfg["radii"], [5], N, cfg["j0"], cfg["B_N"], with_k=False) \
        .to_csv(out_dir / "Experiment1_Constants.csv", index=False)
    jobs = [(cfg, a, is_ + 1, iq + 1, s, snr, out_dir)            # 1-based is/iq as in MATLAB seeds
            for is_, s in enumerate(cfg["signals"]) for iq, snr in enumerate(cfg["snrs"])]
    t0 = time.time()
    results = run_conditions(e1_condition_job, jobs, n_jobs)
    overall, gains, ranking = e1_compile_tables(results, cfg, names)
    overall["N"] = N; overall["RMSE"] = np.sqrt(overall["AMSE"])
    overall.to_csv(out_dir / "Experiment1_AMSE_all.csv", index=False)
    gains.to_csv(out_dir / "Experiment1_PairedGains.csv", index=False)
    ranking.to_csv(out_dir / "Experiment1_AggregateRanking.csv", index=False)
    save_pickle(dict(cfg=cfg, a=a, results=results), out_dir / "Experiment1_Results.pkl")
    (out_dir / "Experiment1_README.txt").write_text(
        f"SCOPE Experiment I | fixed k=5 B_N={cfg['B_N']:.5g} | j0={cfg['j0']} n={N} M={Mrep}\n"
        f"Signals {', '.join(cfg['signals'])}; root SNR=std(f,1)/std(eps)={cfg['snrs']}; eps iid N(0,1).\n"
        "SCOPE = r0; N-SCOPE = r1,r2,r3; 6 centered CDFs; universal hard/soft references.\n"
        f"Calibration: R_F,r,k(a_j)=B_N/2^j; sigmaHat = MAD(finest)/0.6745; paired noise.\n"
        "Python translation of SCOPE_Exp123_BN: NumPy MT19937 noise (not bit-identical to MATLAB randn).\n")
    log(f"Experiment I finished: {out_dir} ({time.time() - t0:.1f} s)")
    return out_dir

## Part 5 — Experiment II: SURE selection of attenuation and neighbourhood

For each generator $F$ the candidate set is the full grid
$k\in\{3,5,7\}$, $r\in\{0,1,2,3\}$, $b\in\{0.65,0.8,1,1.2,1.5\}$ (60
candidates, ordered radius → $k$ → $b$, as in `e2_make_candidates`).
Per replication and generator, four procedures are evaluated:

| Method | Rule |
|---|---|
| `FixedSCOPE` | $(k,r,b)=(5,0,1)$ |
| `FixedNSCOPE` | $(k,r,b)=(5,1,1)$ |
| `SCOPE_SURE` | minimise global SURE over $r=0$ candidates |
| `NSCOPE_SURE` | minimise global SURE over all 60 candidates |

SCOPE SURE is nested inside N-SCOPE SURE, so the minimum *estimated* risk
of N-SCOPE SURE can never be larger (checked by an assertion); its realised
MSE can still be larger because selection overfits. The true coefficients
are used **only after** the selection, to score the chosen estimate.
Batches of 20 replications; same seeds per condition as Experiment I.

In [ ]:
def e2_make_candidates(kgrid, rgrid, bgrid):
    rows = [(ir, r, ik, k, b) for ir, r in enumerate(rgrid) for ik, k in enumerate(kgrid) for b in bgrid]
    cand = {key: np.array([row[i] for row in rows]) for i, key in enumerate(["ir", "r", "ik", "k", "b"])}
    cand["scopeIndices"] = np.flatnonzero(cand["r"] == 0)
    fs = np.flatnonzero((cand["r"] == 0) & (cand["k"] == 5) & (np.abs(cand["b"] - 1) < 1e-12))
    fn = np.flatnonzero((cand["r"] == 1) & (cand["k"] == 5) & (np.abs(cand["b"] - 1) < 1e-12))
    assert fs.size == 1 and fn.size == 1
    cand["fixedScope"], cand["fixedNScope"] = int(fs[0]), int(fn[0])
    return cand

def e2_method_names(generators):
    suffix = ["FixedSCOPE", "FixedNSCOPE", "SCOPE_SURE", "NSCOPE_SURE"]
    return [f"{F}_{s}" for F in generators for s in suffix] + ["UniversalHard", "UniversalSoft"]

def e2_evaluate_selected(db, tr, S, err_scaling, sig, F, a, iF, cand, idx, N):
    """Squared error of the SELECTED candidate per replication (truth used only here)."""
    loss = err_scaling.copy()
    for jj in range(len(db)):
        dj, theta = db[jj], np.zeros_like(db[jj])
        for c in np.unique(idx):                       # group replications by candidate
            cols = np.flatnonzero(idx == c)
            lam = cand["b"][c] * a[iF, cand["ik"][c], cand["ir"][c], jj] / sig[cols]
            theta[:, cols] = dj[:, cols] * cdf_centered(F, S[jj][cand["ir"][c]][:, cols] * lam) ** cand["k"][c]
        loss += np.sum((theta - tr[jj][:, None]) ** 2, axis=0)
    return loss / N

def e2_simulate_condition(f, W, a, cfg, cand, names, tag, filter_name, snr, partial_fn, seed):
    N, Mrep, j0 = cfg["N"], cfg["M"], cfg["j0"]
    nF, nC, nmeth = len(cfg["generators"]), len(cand["k"]), len(names)
    if cfg["resume"] and Path(partial_fn).exists():
        st = load_pickle(partial_fn)
        assert st["M"] == Mrep and st["nF"] == nF and st["nC"] == nC and st["settings"] == cfg["settings"], \
            f"Incompatible partial checkpoint. Move/delete {partial_fn}"
        rng = make_rng(0); rng.bit_generator.state = st["rngState"]
        log(f"  Resuming PARTIAL at replicate {st['completed']}/{Mrep}.")
    else:
        rng = make_rng(seed)
        st = dict(M=Mrep, nF=nF, nC=nC, settings=cfg["settings"], completed=0,
                  mse=np.full((Mrep, nmeth), np.nan), sigmahat=np.full(Mrep, np.nan),
                  scopeIdx=np.zeros((Mrep, nF), int), nscopeIdx=np.zeros((Mrep, nF), int),
                  firstReplication=None)
    trueD = W @ f
    lv = level_slices(N, j0)
    for istart in range(st["completed"], Mrep, cfg["batchSize"]):
        iend = min(Mrep, istart + cfg["batchSize"]); nb = iend - istart
        Y = f[:, None] + rng.standard_normal((N, nb))
        D = W @ Y
        sig = sigma_mad(D)
        assert np.all(sig > 0) and np.all(np.isfinite(sig))
        st["sigmahat"][istart:iend] = sig
        if istart == 0:
            st["firstReplication"] = dict(y=Y[:, 0].copy(), d=D[:, 0].copy(), sigmahat=sig[0])
        db = [D[sl] for _, sl in lv]
        tr = [trueD[sl] for _, sl in lv]
        S = [[local_rms(dj, r) for r in cfg["rGrid"]] for dj in db]
        err_scaling = np.sum((D[:2 ** j0] - trueD[:2 ** j0, None]) ** 2, axis=0)
        mse_b = np.full((nb, nmeth), np.nan)
        for iF, F in enumerate(cfg["generators"]):
            sure = np.zeros((nC, nb))
            for c in range(nC):                           # every candidate, summed over levels
                ir, ik, k, b = cand["ir"][c], cand["ik"][c], cand["k"][c], cand["b"][c]
                mm = 2 * cand["r"][c] + 1
                for jj in range(len(db)):
                    dj, Sj = db[jj], S[jj][ir]
                    lam = b * a[iF, ik, ir, jj] / sig
                    g, gp = cdf_centered(F, Sj * lam, deriv=True)
                    w = g ** k
                    corr = (k / mm) * (dj ** 2 / np.maximum(Sj, REALMIN)) * g ** (k - 1) * gp
                    deriv = w + corr * lam
                    sure[c] += (np.sum((dj * (w - 1)) ** 2, axis=0) + 2 * sig ** 2 * np.sum(deriv, axis=0)
                                - dj.shape[0] * sig ** 2)
            assert np.all(np.isfinite(sure)), "Non-finite plug-in SURE encountered in candidate search."
            loc = np.argmin(sure[cand["scopeIndices"]], axis=0)       # first minimum, as MATLAB min
            idx_s = cand["scopeIndices"][loc]
            min_s = sure[cand["scopeIndices"]][loc, np.arange(nb)]
            idx_n = np.argmin(sure, axis=0); min_n = sure[idx_n, np.arange(nb)]
            assert np.all(min_n <= min_s + 1e-7 * np.maximum(1, np.abs(min_s))), "Nested SURE check failed."
            st["scopeIdx"][istart:iend, iF] = idx_s
            st["nscopeIdx"][istart:iend, iF] = idx_n
            choices = [np.full(nb, cand["fixedScope"]), np.full(nb, cand["fixedNScope"]), idx_s, idx_n]
            for meth, ch in enumerate(choices):
                mse_b[:, iF * 4 + meth] = e2_evaluate_selected(db, tr, S, err_scaling, sig, F, a, iF, cand, ch, N)
        thr = sig * np.sqrt(2 * np.log(N))
        hard_l, soft_l = err_scaling.copy(), err_scaling.copy()
        for dj, t in zip(db, tr):
            hard_l += np.sum((dj * (np.abs(dj) > thr) - t[:, None]) ** 2, axis=0)
            soft_l += np.sum((np.sign(dj) * np.maximum(np.abs(dj) - thr, 0) - t[:, None]) ** 2, axis=0)
        mse_b[:, -2], mse_b[:, -1] = hard_l / N, soft_l / N
        assert np.all(np.isfinite(mse_b)) and np.all(mse_b >= 0)
        st["mse"][istart:iend] = mse_b
        st["completed"] = iend
        st["rngState"] = rng.bit_generator.state
        save_pickle(st, partial_fn)
        log(f"   E2 {tag} / SNR {snr}: {iend:4d}/{Mrep:4d} ({100 * iend / Mrep:5.1f}%)")
    assert st["completed"] == Mrep and np.all(np.isfinite(st["mse"]))
    return dict(signal=tag, snr=snr, filter=filter_name, M=Mrep, names=names, mse=st["mse"],
                sigmahat=st["sigmahat"], scopeIdx=st["scopeIdx"], nscopeIdx=st["nscopeIdx"],
                cand=cand, firstReplication=st["firstReplication"], waveletShift=cfg["shift"],
                studySettings=cfg["settings"])

def e2_write_selection_csv(R, path):
    T = pd.DataFrame({"Replication": np.arange(1, R["M"] + 1), "SigmaHat": R["sigmahat"]})
    for fi in range(R["scopeIdx"].shape[1]):
        tag = R["names"][4 * fi][:-len("_FixedSCOPE")]
        for method, ids in (("SCOPE", R["scopeIdx"][:, fi]), ("NSCOPE", R["nscopeIdx"][:, fi])):
            for p in ("k", "r", "b"):
                T[f"{tag}_{method}_{p}"] = R["cand"][p][ids]
    T.to_csv(path, index=False)

def e2_compile_tables(results, cfg, names):
    mtypes = ["FixedSCOPE", "FixedNSCOPE", "SCOPE_SURE", "NSCOPE_SURE"]
    overall, paired, sel, triple, relative = [], [], [], [], []
    nF = len(cfg["generators"])
    for R in results:
        Mrep, means, sd = R["M"], R["mse"].mean(axis=0), R["mse"].std(axis=0, ddof=1)
        relative.append(means / means[-2])                 # common HARD benchmark
        for fi, F in enumerate(cfg["generators"]):
            ind = [fi * 4 + m for m in range(4)]
            for meth, ii in enumerate(ind):
                gain = 100 * (means[ind[0]] - means[ii]) / means[ind[0]]
                overall.append([R["signal"], R["snr"], R["filter"], F, mtypes[meth], names[ii],
                                Mrep, means[ii], sd[ii], sd[ii] / np.sqrt(Mrep), gain])
            for A, B in [(3, 2), (2, 0), (3, 1), (3, 0), (1, 0)]:   # MATLAB [4 3;3 1;4 2;4 1;2 1]
                iA, iB = ind[A], ind[B]
                delta = R["mse"][:, iB] - R["mse"][:, iA]          # > 0: A has lower AMSE
                dm, se = delta.mean(), delta.std(ddof=1) / np.sqrt(Mrep)
                paired.append([R["signal"], R["snr"], F, names[iA], names[iB], dm, se,
                               dm - 1.96 * se, dm + 1.96 * se, 100 * dm / means[iB]])
            for kind, ids, mt in ((1, R["scopeIdx"][:, fi], "SCOPE_SURE"), (2, R["nscopeIdx"][:, fi], "NSCOPE_SURE")):
                for var, vals in (("k", cfg["kGrid"]), ("r", cfg["rGrid"]), ("b", cfg["bGrid"])):
                    cv = R["cand"][var][ids]
                    for v in vals:
                        n = int(np.sum(np.abs(cv - v) < 1e-9))
                        sel.append([R["signal"], R["snr"], F, mt, var, v, n, n / Mrep])
                for c in range(len(R["cand"]["k"])):
                    if kind == 1 and R["cand"]["r"][c] != 0:
                        continue
                    n = int(np.sum(ids == c))
                    if n > 0:
                        triple.append([R["signal"], R["snr"], F, mt, R["cand"]["k"][c], R["cand"]["r"][c],
                                       R["cand"]["b"][c], n, n / Mrep])
        for ref in range(2):
            ii = nF * 4 + ref
            overall.append([R["signal"], R["snr"], R["filter"], "Universal", "Reference", names[ii],
                            Mrep, means[ii], sd[ii], sd[ii] / np.sqrt(Mrep), np.nan])
    overall = pd.DataFrame(overall, columns=["Signal", "SNR", "Filter", "Generator", "Method", "Procedure",
                                             "M", "AMSE", "SD_MSE", "SE_AMSE", "GainPct_vs_FixedSCOPE"])
    paired = pd.DataFrame(paired, columns=["Signal", "SNR", "Generator", "Procedure_A", "Procedure_B",
                                           "Delta_BminusA", "PairedSE", "CI95_Lower", "CI95_Upper", "GainPct_A_over_B"])
    sel = pd.DataFrame(sel, columns=["Signal", "SNR", "Generator", "Method", "Parameter", "Value", "Count", "Frequency"])
    triple = pd.DataFrame(triple, columns=["Signal", "SNR", "Generator", "Method", "K", "R", "B", "Count", "Frequency"])
    rel = np.mean(relative, axis=0); order = np.argsort(rel, kind="stable")
    ranking = pd.DataFrame({"Procedure": np.array(names)[order], "MeanRelativeAMSE_to_UniversalHard": rel[order]})
    return overall, paired, sel, triple, ranking

def e2_condition_job(cfg, a, cand, is_, iq, sig_name, snr, out_dir):
    f0, filt, which = make_dj_signal(sig_name, cfg["N"])
    W = wavelet_bank(cfg["N"], cfg["j0"], cfg["shift"])[0][which]
    names = e2_method_names(cfg["generators"])
    cp = out_dir / f"checkpoint_{sig_name}_SNR{fmt_snr(snr)}.pkl"
    partial = out_dir / f"PARTIAL_{sig_name}_SNR{fmt_snr(snr)}.pkl"
    if cfg["resume"] and cp.exists():
        R = load_pickle(cp)
        assert R["mse"].shape == (cfg["M"], len(names)) and R["studySettings"] == cfg["settings"], \
            "Existing completed checkpoint is incompatible with the settings."
        log(f"RESUMED completed E2: {sig_name} SNR={snr}")
    else:
        f = scale_to_snr(f0, snr)
        log(f"Starting E2 {sig_name} | SNR={snr} | M={cfg['M']}")
        R = e2_simulate_condition(f, W, a, cfg, cand, names, sig_name, filt, snr, partial,
                                  cfg["seed"] + 100 * is_ + iq)
        save_pickle(R, cp)
        if partial.exists(): partial.unlink()
    write_mse_csv(R, out_dir / f"MSE_replications_{sig_name}_SNR{fmt_snr(snr)}.csv")
    e2_write_selection_csv(R, out_dir / f"SelectedParams_{sig_name}_SNR{fmt_snr(snr)}.csv")
    return R

def run_experiment2(mode, N, Mrep, snrs, out_base, snr_tag, n_jobs=1, resume=True):
    smoke = mode == "smoke"
    B_N, j0 = scope_B_for_N(N)
    cfg = dict(N=N, J=int(round(np.log2(N))), j0=j0, B_N=B_N, kGrid=[3, 5, 7], rGrid=[0, 1, 2, 3],
               bGrid=[0.65, 0.80, 1.00, 1.20, 1.50], generators=list(GENERATORS),
               signals=list(SIGNALS), snrs=list(snrs), snrDefinition="std_signal_over_std_noise_iidN01",
               M=Mrep, batchSize=20, seed=20261026 + N, shift=2, resume=resume)
    if smoke:
        cfg.update(generators=["Normal", "Logistic"], signals=["Doppler", "Blocks"], snrs=[5],
                   batchSize=min(3, Mrep))
        out_dir = Path(out_base) / f"Exp2_N{N}_SMOKE"
    else:
        out_dir = Path(out_base) / f"Exp2_N{N}_M{Mrep}{snr_tag}"
    out_dir.mkdir(parents=True, exist_ok=True)
    cfg["settings"] = {k: cfg[k] for k in ("N", "M", "J", "j0", "B_N", "kGrid", "rGrid", "bGrid",
                                          "generators", "batchSize", "seed", "shift", "snrDefinition")}
    cand = e2_make_candidates(cfg["kGrid"], cfg["rGrid"], cfg["bGrid"])
    validate_sure_divergence()
    names = e2_method_names(cfg["generators"])
    log(f"Experiment II ({mode}), N={N}: M={Mrep}, {len(cfg['generators'])} CDFs, {len(cand['k'])} candidates per CDF.")
    a = compute_constants(N, cfg["generators"], cfg["rGrid"], cfg["kGrid"], cfg["j0"], cfg["B_N"],
                          cache=out_dir / "calibration_E2.pkl")
    check_constants(a, cfg["generators"], cfg["rGrid"], cfg["kGrid"], N, cfg["B_N"])
    constants_table(a, cfg["generators"], cfg["rGrid"], cfg["kGrid"], N, cfg["j0"], cfg["B_N"]) \
        .to_csv(out_dir / "Experiment2_ScaleConstants.csv", index=False)
    jobs = [(cfg, a, cand, is_ + 1, iq + 1, s, snr, out_dir)
            for is_, s in enumerate(cfg["signals"]) for iq, snr in enumerate(cfg["snrs"])]
    t0 = time.time()
    results = run_conditions(e2_condition_job, jobs, n_jobs)
    summary, paired, sel, triple, ranking = e2_compile_tables(results, cfg, names)
    summary["N"] = N; summary["RMSE"] = np.sqrt(summary["AMSE"])
    summary.to_csv(out_dir / "Experiment2_AMSE_all.csv", index=False)
    paired.to_csv(out_dir / "Experiment2_PairedDifferences.csv", index=False)
    sel.to_csv(out_dir / "Experiment2_SelectionFrequencies.csv", index=False)
    triple.to_csv(out_dir / "Experiment2_SelectedTriples.csv", index=False)
    ranking.to_csv(out_dir / "Experiment2_AggregateRanking.csv", index=False)
    save_pickle(dict(cfg=cfg, a=a, cand=cand, results=results), out_dir / "Experiment2_Results.pkl")
    (out_dir / "Experiment2_README.txt").write_text(
        f"SCOPE Experiment II | n={N}, M={Mrep}, sigma=1 | signals {', '.join(cfg['signals'])} | root SNR {cfg['snrs']}\n"
        f"CDFs {', '.join(cfg['generators'])}; kGrid={cfg['kGrid']} rGrid={cfg['rGrid']} bGrid={cfg['bGrid']}\n"
        f"Global SURE tuning across detail levels j={j0},...,{cfg['J'] - 1}; "
        f"lambda_j=b*R^(-1)_F,r,k(B_N/2^j)/sigmahat, B_N={B_N:g}.\n"
        "Plug-in SURE (sigmahat fixed) is not exactly unbiased; error measured after selection.\n"
        "Python translation of SCOPE_Exp123_BN: NumPy MT19937 noise (not bit-identical to MATLAB randn).\n")
    log(f"Experiment II finished: {out_dir} ({time.time() - t0:.1f} s)")
    return out_dir

## Part 6 — Experiment III: frozen SCOPE recipes against 20 comparators

Four **Normal-generator** SCOPE recipes were frozen *before* this
independent Monte Carlo (new seeds `20271126 + N`), using the Experiment
I/II rankings:

| # | Procedure | $(k, r, b)$ |
|---|---|---|
| 1 | Fixed SCOPE | $(5, 0, 1)$ |
| 2 | Fixed N-SCOPE | $(5, 1, 1)$ |
| 3 | SCOPE SURE | global SURE over $k\in\{3,5,7\}$, $b$-grid, $r=0$ |
| 4 | N-SCOPE SURE | global SURE over $k$, $r\in\{0,\dots,3\}$, $b$ |

Methods 5–24 are the literature comparators (Table 3 of the paper, same
order), and 25–26 the universal hard/soft references. Every method
receives the **same** $d = Wy$ and the same $\hat\sigma$ (MAD) unless its
published definition estimates the noise itself (BAMS, Epanechnikov).
All methods leave the 8 scaling coefficients untouched and shrink levels
$j = 3,\dots,J-1$.

**Audit status** (see `METHOD_IMPLEMENTATION_NOTES.csv`): several image-domain
methods are one-dimensional adaptations; EB-block and HMT are provisional
implementations, not reproductions of the original research code.

Below, each comparator is a separate, documented function. Level vectors are
1-D NumPy arrays; "periodic" neighbourhoods wrap around within a level.

### Shared helpers
Standard normal $\Phi$, $\Phi^{-1}$, $\phi$ without toolboxes; soft
thresholding $\eta_T(x)=\operatorname{sign}(x)(|x|-T)_+$; periodic moving
averages; 64-point Gauss–Legendre quadrature (used by the raised-cosine and
Epanechnikov Bayes rules; identical nodes to MATLAB's Golub–Welsch routine).

In [ ]:
def Phi0(x): return 0.5 * erfc(-x / np.sqrt(2))
def PhiInv0(p): return -np.sqrt(2) * erfcinv(2 * np.clip(p, 1e-15, 1 - 1e-15))
def phi0(x): return np.exp(-0.5 * x ** 2) / np.sqrt(2 * np.pi)
def soft_threshold(x, T): return np.sign(x) * np.maximum(np.abs(x) - T, 0)

def periodic_local_mean(x, w):
    """Centred periodic moving average, odd window w (periodic_local_mean / local_average)."""
    r = (w - 1) // 2
    return sum(np.roll(x, k) for k in range(-r, r + 1)) / w

GL_X, GL_W = np.polynomial.legendre.leggauss(64)

def empirical_quantile_sorted(xs, p):
    """Linear-interpolation quantile at r = 1 + (n-1)p of a sorted vector."""
    n = xs.size
    if p <= 0: return xs[0]
    if p >= 1: return xs[-1]
    r = (n - 1) * p
    i1, i2 = int(np.floor(r)), int(np.ceil(r))
    return xs[i1] if i1 == i2 else (1 - (r - i1)) * xs[i1] + (r - i1) * xs[i2]

### #5 BAMS — author's MATLAB variant (Vidakovic & Ruggeri, 2001)
Model: $d\mid\theta,s^2\sim N(\theta,s^2)$, $s^2\sim\mathrm{Exp}(\mu)$, so
$d\mid\theta\sim\mathrm{DE}(\sqrt{2\mu})$; prior
$\theta\sim\varepsilon_j\delta_0+(1-\varepsilon_j)\mathrm{DE}(0,\tau_j)$;
rule = posterior mean. **Author settings reproduced exactly**
(`scope_BAMS_author_coefficients`):
pseudo-$s$ = IQR of the finest level / $C$ with $C=2$, $\mu = 1/s^2$;
$\tau_u = \sqrt{\max(\operatorname{Var}(d) - 1/\mu,\,10^{-4})}$ over **all**
coefficients; $\tau_j = \tau_u\sqrt j$; $\varepsilon_j = 1 - (j-j_0+1)^{-\gamma}$,
$\gamma = 2$; decomposition depth $L = J-3$ (the corrected depth).
The closed form `bams_bayesrule` is vectorised here with identical algebra.
*Note:* that closed form is singular at $\tau_j^2 = 1/(2\mu)$ (see audit);
the author version is kept for fidelity.

In [ ]:
def bams_bayesrule(d, mu, tau, eps_):
    """Posterior mean of the eps-contaminated DE/DE model (vectorised bams_bayesrule)."""
    s2m = np.sqrt(2 * mu)
    de = 0.5 * s2m * np.exp(-np.abs(d) * s2m)
    m = (tau * np.exp(-np.abs(d) / tau) - 1 / s2m * np.exp(-s2m * np.abs(d))) / (2 * tau ** 2 - 1 / mu)
    c = tau ** 2 - 1 / (2 * mu)
    br_pos = (tau * c * d * np.exp(-d / tau) + tau ** 2 / mu * (np.exp(-s2m * d) - np.exp(-d / tau))) / \
             (c * (tau * np.exp(-d / tau) - 1 / s2m * np.exp(-s2m * d)))
    br_neg = (tau * c * d * np.exp(d / tau) - tau ** 2 / mu * (np.exp(s2m * d) - np.exp(d / tau))) / \
             (c * (tau * np.exp(d / tau) - 1 / s2m * np.exp(s2m * d)))
    br = np.where(d >= 0, br_pos, br_neg)
    return (1 - eps_) * m * br / ((1 - eps_) * m + eps_ * de)

def bams_author_coefficients(d, J, L, Cscale=2.0, gam=2.0):
    sw = np.asarray(d, float).copy(); n = sw.size; coarsest = J - L
    assert 2 ** J == n and L >= 1 and 0 <= coarsest < J, "BAMS requires a valid decomposition DEPTH L"
    vals = np.sort(sw[n // 2:]); mm = vals.size
    lo, hi = mm // 4, (3 * mm) // 4                       # MATLAB floor(mm/4), floor(3mm/4), 1-based
    pseudos = abs(vals[hi - 1] - vals[lo - 1]) / Cscale
    mu = 1 / max(pseudos ** 2, REALMIN)
    tauu = np.sqrt(max(np.std(sw, ddof=1) ** 2 - 1 / mu, 0.0001))
    out = sw.copy()
    for j in range(J - 1, coarsest - 1, -1):
        epsj = 1 - 1 / (j - coarsest + 1) ** gam
        sl = slice(2 ** j, 2 ** (j + 1))
        out[sl] = bams_bayesrule(sw[sl], mu, tauu * np.sqrt(j), epsj)
    return out

### #6 EBayesThresh posterior median (Johnstone & Silverman, 2005)
Per level, standardise $z = d/\hat\sigma$; prior
$(1-w)\delta_0 + w\,\gamma_a$ with Laplace slab $\gamma_a(\theta)=\tfrac a2e^{-a|\theta|}$,
$a = 0.5$ (EbayesThresh default). $w$ is the marginal maximum-likelihood
estimate, restricted so that the posterior-median threshold does not exceed
$\sqrt{2\log n}$; the score $\sum (g-\phi)/((1-w)\phi+wg)$ is monotone in $w$
and solved by 60 bisection steps. The estimate is the exact posterior median.

In [ ]:
def laplace_marginal_parts(x, a):
    """phi(x), g = Laplace*Normal convolution and its theta>0 / theta<0 parts, x>=0."""
    x = np.abs(x)
    ph = phi0(x)
    Ap = 0.5 * a * np.exp(0.5 * a ** 2 - a * x) * Phi0(x - a)
    Am = 0.5 * a * np.exp(0.5 * a ** 2 + a * x) * Phi0(-x - a)
    Ap = np.where(np.isfinite(Ap), Ap, 0.0); Am = np.where(np.isfinite(Am), Am, 0.0)
    g = np.maximum(Ap + Am, REALMIN)
    return ph, g, Ap, Am

def ebayes_fit_w(z, a):
    n = z.size; x = np.abs(z)
    ph, g, _, _ = laplace_marginal_parts(x, a)
    tU = np.sqrt(2 * np.log(max(n, 2)))
    phT, _, ApT, AmT = laplace_marginal_parts(np.array(tU), a)
    denT = phT + (ApT - AmT)
    wmin = 1e-8 if (denT <= 0 or not np.isfinite(denT)) else float(phT / denT)
    wmin = min(max(wmin, 1e-8), 1 - 1e-8)
    score = lambda w: np.sum((g - ph) / np.maximum((1 - w) * ph + w * g, REALMIN))
    if score(wmin) <= 0:
        w = wmin
    elif score(1 - 1e-10) >= 0:
        w = 1 - 1e-10
    else:
        lo, hi = wmin, 1 - 1e-10
        for _ in range(60):
            mid = 0.5 * (lo + hi)
            if score(mid) > 0: lo = mid
            else: hi = mid
        w = 0.5 * (lo + hi)
    return w, float(np.sum(np.log(np.maximum((1 - w) * ph + w * g, REALMIN))))

def ebayes_postmed_laplace(z, w, a):
    sgn, x = np.sign(z), np.abs(z)
    ph, g, Ap, _ = laplace_marginal_parts(x, a)
    wpost = w * g / np.maximum((1 - w) * ph + w * g, REALMIN)
    is_zero = wpost * (Ap / np.maximum(g, REALMIN)) <= 0.5
    mabs = np.zeros_like(x)
    idx = (~is_zero) & (x > 0)
    if np.any(idx):
        xx, wp = x[idx], wpost[idx]
        Dv = np.exp(-a * xx) * Phi0(xx - a) + np.exp(a * xx) * Phi0(-xx - a)
        u = np.clip(Dv * np.exp(a * xx) / (2 * wp), 1e-14, 1 - 1e-14)
        mabs[idx] = np.maximum(xx - a - PhiInv0(u), 0)
    return sgn * mabs

def ebayes_laplace_median_level(d, sigma, a=0.5):
    z = d / sigma
    w, _ = ebayes_fit_w(z, a)
    return sigma * ebayes_postmed_laplace(z, w, a)

### #7 Raised-cosine Bayes (Reina & Sousa) and #8 Epanechnikov Bayes (Barrios & Sousa)
Both use the spike-and-slab prior $\alpha_j\delta_0 + (1-\alpha_j)g$ and the
posterior mean, with $\alpha_j = 1-(j-j_0+1)^{-2}$ (the papers' level rule
with $\gamma=2$, $l=1$), evaluated by 64-point Gauss–Legendre quadrature on
the bounded slab support.

* **Raised cosine:** $g(\theta)=\frac{1}{2\tau}[1+\cos(\pi\theta/\tau)]$ on
  $(-\tau,\tau)$, $\tau = \max_{j,k}|d_{j,k}|$; Gaussian likelihood with $\hat\sigma$.
* **Epanechnikov:** $g(\theta)=\frac{3}{4\beta}(1-\theta^2/\beta^2)$ on
  $(-\beta,\beta)$, $\beta_j=\max_k|d_{j,k}|$; the variance has an
  $\mathrm{Exp}(\lambda)$ prior, which makes the likelihood double exponential
  with rate $\sqrt{2\lambda}$, $\lambda = 1/s^2 + \tfrac12 e^{-s/2}$ and $s$ the
  sample SD of the finest level (the paper's rule with $c=1$, $\tau=2$).

In [ ]:
def raised_cosine_bayes_level(d, sigma, alpha, tau):
    if tau <= 1e-12:
        return np.zeros_like(d)
    theta = tau * GL_X[:, None]; qw = tau * GL_W[:, None]
    gprior = (1 + np.cos(np.pi * theta / tau)) / (2 * tau)
    E = np.exp(-0.5 * ((theta - d[None, :]) / sigma) ** 2)
    slab_m = np.sum(qw * gprior * E, axis=0); slab_n = np.sum(qw * gprior * theta * E, axis=0)
    den = alpha * np.exp(-0.5 * (d / sigma) ** 2) + (1 - alpha) * slab_m
    return (1 - alpha) * slab_n / np.maximum(den, REALMIN)

def epanechnikov_bayes_level(d, alpha, beta, lam):
    if beta <= 1e-12:
        return np.zeros_like(d)
    theta = beta * GL_X[:, None]; qw = beta * GL_W[:, None]
    gprior = (3 / (4 * beta)) * (1 - (theta / beta) ** 2)
    rate = np.sqrt(2 * lam)
    E = np.exp(-rate * np.abs(theta - d[None, :]))
    slab_m = np.sum(qw * gprior * E, axis=0); slab_n = np.sum(qw * gprior * theta * E, axis=0)
    den = alpha * np.exp(-rate * np.abs(d)) + (1 - alpha) * slab_m
    return (1 - alpha) * slab_n / np.maximum(den, REALMIN)

### #9 Smooth SCAD, stabilised level-wise SURE (Kulkarni et al., 2026)
Rule: $0$ for $|d|\le\lambda$; $d - \operatorname{sign}(d)\,\lambda\cos^2\!\big(\tfrac\pi2\,\tfrac{|d|-\lambda}{(a-1)\lambda}\big)$
for $\lambda<|d|<a\lambda$; $d$ for $|d|\ge a\lambda$. Per level, $(a,\lambda)$
minimise SURE over $a\in\{2,2.5,3,3.5\}$ and 122 values $\lambda\in[0,\lambda_U]$,
$\lambda_U=\hat\sigma\sqrt{2\log N}$; then the paper's stabiliser
$\lambda^\star=\min\{\lambda_U,\max(\lambda_S, 2\hat\sigma)\}$ (§7.3).

In [ ]:
def smooth_scad_apply(d, lam, a):
    if lam <= 0:
        return d.copy()
    r = np.abs(d); out = np.zeros_like(d)
    mid = (r > lam) & (r < a * lam); hi = r >= a * lam
    s = (r[mid] - lam) / ((a - 1) * lam)
    out[mid] = d[mid] - np.sign(d[mid]) * lam * np.cos(0.5 * np.pi * s) ** 2
    out[hi] = d[hi]
    return out

def smooth_scad_sure_level(d, sigma, a_grid=(2, 2.5, 3, 3.5), n_lambda=121, c_floor=2, n_global=None):
    n = d.size
    n_global = n if n_global is None else n_global
    lamU = sigma * np.sqrt(2 * np.log(max(n_global, 2)))
    lam_grid = np.r_[0, np.linspace(lamU / n_lambda, lamU, n_lambda)]
    r = np.abs(d); s2 = sigma ** 2
    best, a_hat, lam_sure = np.inf, a_grid[0], 0.0
    for a in a_grid:
        if n * s2 < best:
            best, a_hat, lam_sure = n * s2, a, 0.0
        for lam in lam_grid[1:]:
            low = r <= lam; mid = (r > lam) & (r < a * lam); hi = r >= a * lam
            s = (r[mid] - lam) / ((a - 1) * lam)
            hmag = lam * np.cos(0.5 * np.pi * s) ** 2
            hp = -np.pi / (2 * (a - 1)) * np.sin(np.pi * s)
            val = np.sum(d[low] ** 2 - s2) + np.sum(hmag ** 2 + s2 - 2 * s2 * hp) + hi.sum() * s2
            if val < best:
                best, a_hat, lam_sure = val, a, lam
    lam_hat = min(lamU, max(lam_sure, c_floor * sigma))
    return smooth_scad_apply(d, lam_hat, a_hat)

### #10 SureShrink, hybrid (Donoho & Johnstone, 1995) and #13 BayesShrink (Chang, Yu & Vetterli, 2000)
* **SureShrink:** with $z=d/\hat\sigma$, if $\tfrac1n\sum(z^2-1)\le(\log_2n)^{3/2}/\sqrt n$
  (sparse level) use $\sqrt{2\log n}$; otherwise minimise the exact soft-threshold
  SURE $n-2\#\{|z_i|\le t\}+\sum\min(z_i^2,t^2)$ over $t\in[0,\sqrt{2\log n}]$
  (all knots plus the cap).
* **BayesShrink:** soft threshold $T=\hat\sigma^2/\hat\sigma_X$ with
  $\hat\sigma_X=\sqrt{\max(\overline{d^2}-\hat\sigma^2,0)}$ per level ($T=\infty$ if $\hat\sigma_X=0$).

In [ ]:
def sureshrink_level(d, sigma):
    n = d.size
    z = np.abs(d) / sigma
    cap = np.sqrt(2 * np.log(max(n, 2)))
    if np.mean(z ** 2) - 1 <= np.log2(max(n, 2)) ** 1.5 / np.sqrt(n):
        thr = cap
    else:
        zs = np.sort(z); z2 = zs ** 2; cum2 = np.cumsum(z2)
        k = np.flatnonzero(zs <= cap) + 1                 # 1-based counts
        risk = np.r_[n, n - 2 * k + cum2[k - 1] + (n - k) * z2[k - 1]]
        cand = np.r_[0, zs[k - 1]]
        kc = k.size; below = cum2[kc - 1] if kc else 0.0
        risk = np.r_[risk, n - 2 * kc + below + (n - kc) * cap ** 2]; cand = np.r_[cand, cap]
        thr = cand[np.argmin(risk)]
    return soft_threshold(d, sigma * thr)

def bayesshrink_level(d, sigma):
    sx = np.sqrt(max(np.mean(d ** 2) - sigma ** 2, 0))
    T = np.inf if sx <= np.sqrt(EPS) * max(1, sigma) else sigma ** 2 / sx
    return soft_threshold(d, T)

### #17 Sendur–Selesnick bivariate shrinkage, 1D (BiShrink)
For child $d$ with parent $p$ (same position one level coarser; $p=0$ at the
coarsest detail level): $\hat d = d\,\big(1 - T/\sqrt{d^2+p^2}\big)_+$ with
$T=\sqrt3\,\hat\sigma^2/\hat\sigma_X$, and $\hat\sigma_X$ from a periodic
7-point moving average of $d^2$ (the 1-D analogue of the 7×7 window).

In [ ]:
def bishrink_level(dj, parent, sigma):
    p = np.zeros_like(dj) if parent is None else np.repeat(parent, 2)
    v = np.maximum(periodic_local_mean(dj ** 2, 7) - sigma ** 2, 1e-10)
    s = np.sqrt(np.maximum(v, 1e-10))
    q = np.sqrt(dj ** 2 + p ** 2)
    T = np.sqrt(3) * sigma ** 2 / s
    return dj * np.maximum(1 - T / np.maximum(q, REALMIN), 0)

### The eight "legacy" comparators (MATLAB `E3_legacy_methods`)
Columns 5, 6, 7, 8, 9, 10, 13, 17 of the result table.

In [ ]:
def e3_legacy_methods(d, sigma, j0=3):
    N = d.size; J = int(round(np.log2(N))); L = j0
    lv = level_slices(N, L)
    est = np.zeros((N, 8))
    est[:, 0] = bams_author_coefficients(d, J, J - L, 2, 2)              # #5 BAMS (depth J-3)
    z = d.copy()
    for j, sl in lv: z[sl] = ebayes_laplace_median_level(d[sl], sigma, 0.5)
    est[:, 1] = z                                                       # #6 EBayes
    tau = max(np.max(np.abs(d[2 ** L:])), 1e-8)
    z = d.copy()
    for j, sl in lv:
        alpha = min(max(1 - 1 / (j - L + 1) ** 2, 0), 1 - 1e-12)
        z[sl] = raised_cosine_bayes_level(d[sl], sigma, alpha, tau)
    est[:, 2] = z                                                       # #7 raised cosine
    s_fine = max(np.std(d[N // 2:], ddof=1), 1e-8)
    lam = 1 / s_fine ** 2 + 0.5 * np.exp(-s_fine / 2)
    z = d.copy()
    for j, sl in lv:
        alpha = min(max(1 - 1 / (j - L + 1) ** 2, 0), 1 - 1e-12)
        z[sl] = epanechnikov_bayes_level(d[sl], alpha, max(np.max(np.abs(d[sl])), 1e-8), lam)
    est[:, 3] = z                                                       # #8 Epanechnikov
    z = d.copy()
    for j, sl in lv: z[sl] = smooth_scad_sure_level(d[sl], sigma, (2, 2.5, 3, 3.5), 121, 2, N)
    est[:, 4] = z                                                       # #9 smooth SCAD
    z = d.copy()
    for j, sl in lv: z[sl] = sureshrink_level(d[sl], sigma)
    est[:, 5] = z                                                       # #10 SureShrink
    z = d.copy()
    for j, sl in lv: z[sl] = bayesshrink_level(d[sl], sigma)
    est[:, 6] = z                                                       # #13 BayesShrink
    z = d.copy()
    for j, sl in lv:
        parent = None if j == L else d[2 ** (j - 1):2 ** j]
        z[sl] = bishrink_level(d[sl], parent, sigma)
    est[:, 7] = z                                                       # #17 BiShrink
    assert np.all(np.isfinite(est)), "Legacy estimator yielded nonfinite value"
    return est

### #11 FDR, #12 EB-block (provisional), #14 NeighCoeff, #18 NeighBlock, #20 BlockJS
* **FDR** (Abramovich & Benjamini; Abramovich et al., 2006): two-sided
  $p_i = \operatorname{erfc}(|z_i|/\sqrt2)$ over all detail coefficients,
  Benjamini–Hochberg step-up at $q = 0.05$, hard thresholding of rejected nulls.
* **EB-block (provisional 1-D proxy, not ABS 2002):** blocks of 4, spike vs
  Gaussian slab $N(0,\tau^2 I)$, $\tau^2$ by moments, ad-hoc $\pi$, posterior mean.
* **NeighCoeff** (Cai & Silverman, 2001): $S^2 = d_{k-1}^2+d_k^2+d_{k+1}^2$,
  $\hat d_k = d_k(1-2\hat\sigma^2\log N/S^2)_+$.
* **NeighBlock** (Cai & Silverman, 2001): blocks $L_0=\lfloor\log N/2\rfloor$
  extended by $L_1=\max(1,\lfloor L_0/2\rfloor)$ on each side, $L=L_0+2L_1$,
  factor $(1-\lambda_*L\hat\sigma^2/S^2)_+$, $\lambda_* = 4.50524$.
* **BlockJS** (Cai, 1999): non-overlapping blocks of $L=\lfloor\log N\rfloor$,
  factor $(1-\lambda_* m\hat\sigma^2/S^2)_+$.

In [ ]:
LAMBDA_STAR = 4.50524      # root of lambda - log(lambda) = 3

def fdr_bh(d, sigma, L=3, q=0.05):
    out = d.copy()
    xx = d[2 ** L:]
    p = erfc((np.abs(xx) / max(sigma, EPS)) / np.sqrt(2))
    ps = np.sort(p); K = ps.size
    ok = np.flatnonzero(ps <= q * np.arange(1, K + 1) / K)
    out[2 ** L:] = 0.0 if ok.size == 0 else xx * (p <= ps[ok[-1]])
    return out

def ebblock_provisional(x, sigma2):
    n = x.size; m = 4
    tau2 = max(np.mean(x ** 2) - sigma2, 1e-5 * sigma2)
    pi1 = min(0.95, max(0.05, np.mean(x ** 2 > 2 * sigma2)))
    z = x.copy()
    for i in range(0, n, m):
        a = x[i:i + m]; mm = a.size
        b0 = np.exp(-np.sum(a ** 2) / (2 * sigma2)) / sigma2 ** (mm / 2)
        b1 = np.exp(-np.sum(a ** 2) / (2 * (sigma2 + tau2))) / (sigma2 + tau2) ** (mm / 2)
        post = pi1 * b1 / (pi1 * b1 + (1 - pi1) * b0 + REALMIN)
        z[i:i + m] = a * post * tau2 / (sigma2 + tau2)
    return z

def neighcoeff_level(x, sigma2, N):
    local = x ** 2 + np.roll(x ** 2, 1) + np.roll(x ** 2, -1)
    return x * np.maximum(1 - 2 * np.log(N) * sigma2 / np.maximum(local, REALMIN), 0)

def neighblock_cs(x, sigma2, N):
    n = x.size
    L0 = max(1, int(np.floor(np.log(N) / 2))); L1 = max(1, L0 // 2); L = L0 + 2 * L1
    z = np.zeros_like(x)
    for i in range(0, n, L0):
        e = min(n, i + L0)
        ii = np.mod(np.arange(i - L1, i + L0 + L1), n)          # fixed-size periodic window
        SS = np.sum(x[ii] ** 2)
        z[i:e] = max(1 - LAMBDA_STAR * L * sigma2 / max(SS, REALMIN), 0) * x[i:e]
    return z

def block_js(x, sigma2, N):
    n = x.size; L = max(1, int(np.floor(np.log(N))))
    z = np.zeros_like(x)
    for i in range(0, n, L):
        blk = x[i:min(n, i + L)]
        z[i:i + blk.size] = blk * max(1 - LAMBDA_STAR * blk.size * sigma2 / max(np.sum(blk ** 2), REALMIN), 0)
    return z

### #15 NeighShrink–SURE (1-D), #16 SURE-LET (1-D two-term), #19 SureBlock, #21 ABE/Jeffreys, #22 Local Wiener
* **NeighShrink–SURE** (Zhou & Cheng, 2008; NeighShrink of Chen et al.):
  $\hat d_k = d_k(1-T/S_k^2)_+$ with $S_k^2$ the periodic window sum of
  squares; per level SURE picks window $2r+1$, $r\in\{1,2,3\}$, and
  $T=(c\,\hat\sigma\sqrt{2\log N})^2$, $c\in\{0.5,0.75,1,1.25\}$. Divergence:
  $\partial\hat d_k/\partial d_k = w_k + 2Td_k^2/S_k^4$ on active coefficients.
* **SURE-LET** (Luisier, Blu & Unser, 2007), pointwise two-term LET
  $\hat d = a_1 d + a_2\,d\,e^{-d^2/(12\hat\sigma^2)}$ with $(a_1,a_2)$
  solving the linear SURE normal equations (pseudo-inverse if ill-conditioned).
* **SureBlock** (Cai & Zhou, 2009): if $T_d\le\gamma_d=\log_2^{3/2}d/\sqrt d$ use the
  garrote with $\lambda = 2\log d$; else minimise exact block-James–Stein SURE
  over integer $L\le\sqrt d$ and $\lambda\in[\max(L-2,0),\,2L\log d]$, checking
  every breakpoint and interval stationary point (no grid).
* **ABE / Jeffreys** (Figueiredo & Nowak, 2001): $\hat d=(d^2-3\hat\sigma^2)_+/d$.
* **Local Wiener** (Mihçak et al., 1999, ML variant): $\hat d = d\,\hat v/(\hat v+\hat\sigma^2)$,
  $\hat v = (\text{7-point periodic mean of } d^2 - \hat\sigma^2)_+$.

In [ ]:
def neighshrink_sure_1d(x, sigma, Tref):
    s2 = sigma ** 2; best = np.inf; z = np.zeros_like(x)
    for r in (1, 2, 3):
        m = x ** 2 + sum(np.roll(x ** 2, l) + np.roll(x ** 2, -l) for l in range(1, r + 1))
        for c in (0.5, 0.75, 1.0, 1.25):
            T = (c * Tref) ** 2
            w = np.maximum(1 - T / np.maximum(m, REALMIN), 0)
            t = x * w
            act = m > T
            div = np.zeros_like(x); div[act] = w[act] + 2 * T * x[act] ** 2 / m[act] ** 2
            sure = np.sum((t - x) ** 2 + 2 * s2 * div - s2)
            if sure < best:
                best, z = sure, t
    return z

def sure_let_1d(x, sigma):
    s2 = max(sigma ** 2, REALMIN)
    t = x / np.sqrt(s2); e = np.exp(-t ** 2 / 12)
    A = np.c_[x, x * e]
    V = np.c_[np.ones_like(x), e * (1 - t ** 2 / 6)]
    G = A.T @ A; rhs = A.T @ x - s2 * V.sum(axis=0)
    rc = 1 / np.linalg.cond(G, 1) if np.all(np.isfinite(G)) and np.any(G) else 0.0
    coef = np.linalg.pinv(G) @ rhs if rc < 1e-11 else np.linalg.solve(G, rhs)
    return A @ coef

def sureblock_1d(x, sigma2):
    d = x.size
    sigma = max(np.sqrt(sigma2), np.sqrt(REALMIN)); u = x / sigma
    if d < 2:
        return x.copy()
    Td = np.mean(u ** 2) - 1; gamma = np.log2(d) ** 1.5 / np.sqrt(d)
    if Td <= gamma:
        return x * np.maximum(1 - 2 * np.log(d) / np.maximum(u ** 2, REALMIN), 0)
    best, bestL, bestLam = np.inf, 1, 0.0
    for L in range(1, int(np.floor(np.sqrt(d))) + 1):
        nB = int(np.ceil(d / L))
        pad = np.zeros(nB * L); pad[:d] = u ** 2
        energy = pad.reshape(nB, L).sum(axis=1)
        m = np.full(nB, float(L)); m[-1] = d - (nB - 1) * L
        order = np.argsort(energy, kind="stable"); E = energy[order]; m = m[order]
        lo, hi = max(L - 2, 0), 2 * L * np.log(d)
        preE = np.r_[0, np.cumsum(E)]; preM = np.r_[0, np.cumsum(m)]
        invE = 1 / np.maximum(E, REALMIN)
        tailInv = np.r_[np.cumsum(invE[::-1])[::-1], 0]
        tailBeta = np.r_[np.cumsum(((m - 2) * invE)[::-1])[::-1], 0]
        eu, first_rev = np.unique(E[::-1], return_index=True)     # unique(E,'last')
        kidx = (E.size - 1 - first_rev) + 1
        good = (eu >= lo) & (eu <= hi)
        cand = np.r_[lo, eu[good], hi]
        inactive = np.r_[np.sum(E <= lo), kidx[good], np.sum(E <= hi)].astype(int)
        iL = np.maximum(lo, np.r_[0, E]); iH = np.minimum(hi, np.r_[E, np.inf])
        k = np.arange(nB + 1)
        valid = (iH > iL) & (tailInv > 0)
        stat = np.zeros(nB + 1)
        stat[valid] = np.minimum(np.maximum(tailBeta[valid] / tailInv[valid], iL[valid]), iH[valid])
        valid &= (stat < iH) | (iH >= hi)
        cand = np.r_[cand, stat[valid]]; inactive = np.r_[inactive, k[valid]]
        risk = d + preE[inactive] - 2 * preM[inactive] + cand ** 2 * tailInv[inactive] - 2 * cand * tailBeta[inactive]
        i = int(np.argmin(risk))
        if risk[i] < best:
            best, bestL, bestLam = risk[i], L, cand[i]
    z = np.zeros(d)
    for b in range(int(np.ceil(d / bestL))):
        sl = slice(b * bestL, min((b + 1) * bestL, d))
        z[sl] = x[sl] * max(1 - bestLam / max(np.sum(u[sl] ** 2), REALMIN), 0)
    return z

def abe_level(x, sigma2):
    return x * np.maximum(1 - 3 * sigma2 / np.maximum(x ** 2, REALMIN), 0)

def local_wiener(x, sigma2):
    v = np.maximum(periodic_local_mean(x ** 2, 7) - sigma2, 0)
    return x * (v / np.maximum(v + sigma2, REALMIN))

### #23 Chang–Yu–Vetterli context modelling, 1-D (CYV 2000)
Per level: context $Z_k = \hat w^\top|u_k|$ with $u_k$ the absolute values of
$d_{k\pm1}, d_{k\pm2}$ (periodic) and the parent $d_{j-1,\lceil k/2\rceil}$
(no parent at $j_0$), **excluding** $d_k$ itself; $\hat w$ = least squares of
$|d_k|$ on $u_k$. Coefficients are sorted by $Z$; for each, the
$L=\max(50,\lceil0.02n\rceil)$ nearest in context order give
$\hat\sigma_Y^2$, $\hat\sigma_X=\sqrt{\max(\hat\sigma_Y^2-\hat\sigma^2,0)}$ and
the soft threshold $T_k=\hat\sigma^2/\hat\sigma_{X,k}$. Levels with
$n<2p+2$ fall back to BayesShrink (CYV with $L=n$).

In [ ]:
def cyv_context_level(x, parent, sigma):
    n = x.size; ax = np.abs(x)
    U = np.c_[np.roll(ax, 1), np.roll(ax, -1), np.roll(ax, 2), np.roll(ax, -2)]
    if parent is not None:
        U = np.c_[U, np.abs(np.repeat(parent, 2))]
    p = U.shape[1]
    if n < 2 * p + 2:
        return bayesshrink_level(x, sigma)
    G = U.T @ U
    rc = 1 / np.linalg.cond(G, 1)
    w = np.linalg.pinv(U) @ ax if rc < 1e-12 else np.linalg.lstsq(U, ax, rcond=None)[0]
    order = np.argsort(U @ w, kind="stable")
    xs = x[order]
    L = min(n, max(50, int(np.ceil(0.02 * n))))
    cs = np.r_[0, np.cumsum(xs ** 2)]
    i = np.arange(n)
    lo = np.clip(i - L // 2, 0, n - L); hi = lo + L
    sX = np.sqrt(np.maximum((cs[hi] - cs[lo]) / L - sigma ** 2, 0))
    T = np.full(n, np.inf); pos = sX > 0; T[pos] = sigma ** 2 / sX[pos]
    out = np.zeros(n); out[order] = np.sign(xs) * np.maximum(np.abs(xs) - T, 0)
    return out

def cyv_context_coefficients(d, sigma, J, j0=3):
    z = d.copy()
    for j in range(j0, J):
        parent = d[2 ** (j - 1):2 ** j] if j > j0 else None
        z[2 ** j:2 ** (j + 1)] = cyv_context_level(d[2 ** j:2 ** (j + 1)], parent, sigma)
    return z

### #24 Two-state hidden Markov tree, 1-D EM (provisional; after Crouse, Nowak & Baraniuk, 1998)
Each detail coefficient has a hidden state $s\in\{\text{small},\text{large}\}$
with $d\mid s\sim N(0,\hat\sigma^2+\tau_s^2)$; states follow a parent→child
Markov tree (8 roots at $j_0$, two children per node). Fifteen EM iterations
use normalised upward/downward messages; $\pi_0$, the transition matrix $A$
and $\tau_s^2$ are **shared across levels** (a simplification of the original
per-scale parameters). Estimate: $\hat d=d\sum_s P(s\mid\text{data})\,\tau_s^2/(\hat\sigma^2+\tau_s^2)$.

In [ ]:
def hmt_1d_em(d, sigma2, L, J):
    x = [d[2 ** j:2 ** (j + 1)] for j in range(L, J)]
    c = J - L
    tau2 = np.array([0.05, max(np.mean(d[2 ** L:] ** 2) - sigma2, 0.2 * sigma2) * 4])
    pi0 = np.array([0.85, 0.15]); A = np.array([[0.94, 0.06], [0.14, 0.86]])
    Gam = [None] * c
    for _ in range(15):
        Up = [None] * c
        for jj in range(c - 1, -1, -1):                       # upward pass
            obs = x[jj]; v = sigma2 + tau2
            logB = np.vstack([-obs ** 2 / (2 * v[0]) - 0.5 * np.log(v[0]),
                              -obs ** 2 / (2 * v[1]) - 0.5 * np.log(v[1])])
            B = np.exp(logB - logB.max(axis=0))
            if jj < c - 1:
                msg = A @ Up[jj + 1]
                B = B * msg[:, 0::2] * msg[:, 1::2]
            B = B / np.maximum(B.sum(axis=0), REALMIN)
            Up[jj] = np.maximum(B, REALMIN)
        G = pi0[:, None] * Up[0]
        Gam[0] = G / np.maximum(G.sum(axis=0), REALMIN)
        counts = np.zeros((2, 2))
        for jj in range(1, c):                                # downward pass
            B = Up[jj]; parents = np.arange(B.shape[1]) // 2
            oldG = Gam[jj - 1][:, parents]
            denom = A @ B
            xi = np.zeros_like(B)
            for ps in range(2):                               # parent state
                v = oldG[ps][None, :] * A[ps][:, None] * B / np.maximum(denom[ps], REALMIN)
                counts[ps] += v.sum(axis=1)
                xi += v
            Gam[jj] = xi / np.maximum(xi.sum(axis=0), REALMIN)
        pi0 = np.maximum(Gam[0].mean(axis=1), 1e-5); pi0 /= pi0.sum()
        A = np.maximum(counts, 1e-6); A = A / A.sum(axis=1, keepdims=True)
        for s in range(2):
            wsum = sum(np.sum(Gam[jj][s] * x[jj] ** 2) for jj in range(c))
            mass = sum(np.sum(Gam[jj][s]) for jj in range(c))
            tau2[s] = max(wsum / max(mass, 1e-6) - sigma2, 0.01 * sigma2)
        tau2[1] = max(tau2[1], tau2[0] + 0.05 * sigma2)
    z = d.copy()
    for jj in range(c):
        j = L + jj
        shrink = Gam[jj][0] * tau2[0] / (sigma2 + tau2[0]) + Gam[jj][1] * tau2[1] / (sigma2 + tau2[1])
        z[2 ** j:2 ** (j + 1)] = x[jj] * shrink
    return z

### The twelve "extended" comparators (MATLAB `E3_extended_methods`)
Columns 11, 12, 14, 15, 16, 18, 19, 20, 21, 22, 23, 24 of the result table.

In [ ]:
def e3_extended_methods(d, sigma, j0=3):
    N = d.size; J = int(round(np.log2(N))); L = j0
    est = np.tile(d[:, None], (1, 12))
    s2 = max(sigma ** 2, REALMIN)
    Tref = sigma * np.sqrt(2 * np.log(N))
    est[:, 0] = fdr_bh(d, sigma, L, 0.05)                               # #11 FDR
    for j, sl in level_slices(N, L):
        x = d[sl]
        est[sl, 1] = ebblock_provisional(x, s2)                          # #12 EB-block proxy
        est[sl, 2] = neighcoeff_level(x, s2, N)                          # #14 NeighCoeff
        est[sl, 3] = neighshrink_sure_1d(x, sigma, Tref)                 # #15 NeighShrink-SURE
        est[sl, 4] = sure_let_1d(x, sigma)                               # #16 SURE-LET
        est[sl, 5] = neighblock_cs(x, s2, N)                             # #18 NeighBlock
        est[sl, 6] = sureblock_1d(x, s2)                                 # #19 SureBlock
        est[sl, 7] = block_js(x, s2, N)                                  # #20 BlockJS
        est[sl, 8] = abe_level(x, s2)                                    # #21 ABE / Jeffreys
        est[sl, 9] = local_wiener(x, s2)                                 # #22 Local Wiener
    est[:, 10] = cyv_context_coefficients(d, sigma, J, L)                # #23 CYV context
    est[:, 11] = hmt_1d_em(d, s2, L, J)                                  # #24 HMT proxy
    assert np.all(np.isfinite(est)), "Nonfinite extended benchmark output"
    return est

### Experiment III: SCOPE recipes, simulation loop and summaries
`apply_e3` and `select_e3` implement the Normal-generator SCOPE rule and its
global SURE search (same formulas as Part 3). Within a replication every
column of `dh` is an estimate of $\theta = Wf$, and the replication MSE of
all 26 methods is $\|dh_{\cdot m}-\theta\|^2/N$.

In [ ]:
E3_NAMES = ["SCOPE_Fixed_Normal", "NSCOPE_Fixed_Normal_r1", "SCOPE_SURE_Normal", "NSCOPE_SURE_Normal",
            "BAMS_Author_C2_Gamma2", "EBayesMedian", "RaisedCosineBayes", "EpanechnikovBayes",
            "SmoothSCAD_stabilized_author", "SureShrink_Hybrid", "FDR_BH_q05", "EBBlock_1D_proxy",
            "BayesShrink_1D", "NeighCoeff", "NeighShrink_SURE_1D", "SURELET_1D_orthonormal",
            "BiShrink_1D", "NeighBlock", "SureBlock_hybrid_CZ09", "BlockJS",
            "Jeffreys_ABE_FigueiredoNowak", "LocalWiener_1D", "Context_CYV2000_1D", "HMT_1D_EM_proxy",
            "UniversalHard", "UniversalSoft"]
LEG_IDS = [i + 4 for i in (1, 2, 3, 4, 5, 6, 9, 13)]           # 1-based table IDs -> 1-based columns
EXT_IDS = [i + 4 for i in (7, 8, 10, 11, 12, 14, 15, 16, 17, 18, 19, 20)]

def e3_neighborhoods(d, cfg):
    return [[local_rms(d[sl], r) for r in cfg["rGrid"]] for _, sl in level_slices(cfg["N"], cfg["j0"])]

def apply_e3(d, S, sigma, a, cfg, k, r, b):
    ki, ri = cfg["kGrid"].index(k), cfg["rGrid"].index(r)
    out = d.copy()
    for jj, (j, sl) in enumerate(level_slices(cfg["N"], cfg["j0"])):
        out[sl] = d[sl] * cdf_centered("Normal", (b * a[0, ki, ri, jj] / sigma) * S[jj][ri]) ** k
    return out

def select_e3(d, S, sigma, a, cfg, r_grid):
    best, kb, rb, bb = np.inf, np.nan, np.nan, np.nan
    s2 = sigma ** 2
    lv = level_slices(cfg["N"], cfg["j0"])
    for r in r_grid:
        ri = cfg["rGrid"].index(r); m = 2 * r + 1
        for ki, k in enumerate(cfg["kGrid"]):
            for b in cfg["bGrid"]:
                risk = 0.0
                for jj, (j, sl) in enumerate(lv):
                    x = d[sl]; s = S[jj][ri]; lam = b * a[0, ki, ri, jj] / sigma
                    g, gp = cdf_centered("Normal", lam * s, deriv=True)
                    w = g ** k
                    der = w + (k / m) * x ** 2 / np.maximum(s, REALMIN) * g ** (k - 1) * gp * lam
                    risk += np.sum((x * (w - 1)) ** 2 + 2 * s2 * der - s2)
                if risk < best:
                    best, kb, rb, bb = risk, k, r, b
    return kb, rb, bb

def e3_estimates(d, sigma, a, cfg):
    """All 26 estimates (columns) for one observed coefficient vector d."""
    N, j0 = cfg["N"], cfg["j0"]
    S = e3_neighborhoods(d, cfg)
    dh = np.zeros((N, 26))
    dh[:, 0] = apply_e3(d, S, sigma, a, cfg, 5, 0, 1)
    dh[:, 1] = apply_e3(d, S, sigma, a, cfg, 5, 1, 1)
    k0, _, b0 = select_e3(d, S, sigma, a, cfg, [0])
    dh[:, 2] = apply_e3(d, S, sigma, a, cfg, k0, 0, b0)
    k1, r1, b1 = select_e3(d, S, sigma, a, cfg, cfg["rGrid"])
    dh[:, 3] = apply_e3(d, S, sigma, a, cfg, k1, r1, b1)
    dh[:, [c - 1 for c in LEG_IDS]] = e3_legacy_methods(d, sigma, j0)
    dh[:, [c - 1 for c in EXT_IDS]] = e3_extended_methods(d, sigma, j0)
    T = sigma * np.sqrt(2 * np.log(N))
    H = d.copy(); H[2 ** j0:] = d[2 ** j0:] * (np.abs(d[2 ** j0:]) > T)
    Q = d.copy(); Q[2 ** j0:] = soft_threshold(d[2 ** j0:], T)
    dh[:, 24], dh[:, 25] = H, Q
    assert np.all(np.isfinite(dh)), "Non-finite estimator"
    return dh, (k0, b0, k1, r1, b1)

def e3_condition_job(cfg, a, si, iq, sig_name, snr, out_dir):
    N = cfg["N"]
    f0, _, which = make_dj_signal(sig_name, N)
    W = wavelet_bank(N, cfg["j0"], cfg["shift"], 1.35)[0][which]
    f = scale_to_snr(f0, snr)
    cp = out_dir / f"checkpoint_{sig_name}_SNR{fmt_snr(snr)}.pkl"
    log(f"E3 {sig_name} | SNR {snr} | M={cfg['M']} | 26 estimators")
    if cfg["resume"] and cp.exists():
        st = load_pickle(cp)
        assert (st.get("codeRevision") == cfg["codeRevision"] and st["M"] == cfg["M"] and
                st["SNR"] == snr and st["Signal"] == sig_name and st["names"] == E3_NAMES and
                st.get("B_N") == cfg["B_N"] and st.get("j0") == cfg["j0"] and
                st.get("snrDefinition") == cfg["snrDefinition"]), f"E3 checkpoint mismatch: {cp}"
        rng = make_rng(0); rng.bit_generator.state = st["rngState"]
        if st["completed"] < cfg["M"]:
            log(f"  resuming at replicate {st['completed'] + 1}")
    else:
        rng = make_rng(cfg["seed"] + 100 * si + iq)
        Mr = cfg["M"]
        st = dict(mse=np.full((Mr, 26), np.nan), sigmaHat=np.full(Mr, np.nan), kScope=np.full(Mr, np.nan),
                  bScope=np.full(Mr, np.nan), kNScope=np.full(Mr, np.nan), rNScope=np.full(Mr, np.nan),
                  bNScope=np.full(Mr, np.nan), names=list(E3_NAMES), M=Mr, SNR=snr, Signal=sig_name,
                  completed=0, rep1=None, codeRevision=cfg["codeRevision"], B_N=cfg["B_N"],
                  j0=cfg["j0"], snrDefinition=cfg["snrDefinition"])
    truth = W @ f
    t0 = time.time()
    for it in range(st["completed"], cfg["M"]):
        y = f + rng.standard_normal(N)
        d = W @ y
        sigma = float(np.median(np.abs(d[N // 2:])) / MAD_CONST)
        assert sigma > 0 and np.isfinite(sigma)
        st["sigmaHat"][it] = sigma
        if it == 0:
            st["rep1"] = dict(f=f, y=y, sigma=sigma)
        dh, (k0, b0, k1, r1, b1) = e3_estimates(d, sigma, a, cfg)
        st["kScope"][it], st["bScope"][it] = k0, b0
        st["kNScope"][it], st["rNScope"][it], st["bNScope"][it] = k1, r1, b1
        st["mse"][it] = np.sum((dh - truth[:, None]) ** 2, axis=0) / N
        st["completed"] = it + 1
        if (it + 1) % cfg["saveEvery"] == 0 or it + 1 == cfg["M"] or it == 0:
            st["rngState"] = rng.bit_generator.state
            save_pickle(st, cp)
            log(f"  E3 {sig_name} SNR {snr}: {it + 1:4d}/{cfg['M']:4d} ({100 * (it + 1) / cfg['M']:.1f}%); "
                f"{time.time() - t0:.1f} s")
    T = pd.DataFrame({"Replicate": np.arange(1, st["M"] + 1), "SigmaHat": st["sigmaHat"],
                      "ScopeK": st["kScope"], "ScopeB": st["bScope"], "NScopeK": st["kNScope"],
                      "NScopeR": st["rNScope"], "NScopeB": st["bNScope"]})
    for i, nm in enumerate(st["names"]):
        T[nm] = st["mse"][:, i]
    T.to_csv(out_dir / f"MSE_{sig_name}_SNR{fmt_snr(snr)}.csv", index=False)
    return st

def e3_write_audit(path):
    status = ["Published-method-inspired 1D adaptation; audit needed"] * 26
    for i in range(4): status[i] = "SCOPE specification: fixed null risk / plug-in SURE"
    status[4] = "User-supplied BAMS.m posterior mean: Cscale=2,gamma=2,depth=J-3,level tau_j=tau_u sqrt(j)"
    status[5] = "Empirical Bayes median: implementation audit pending"
    status[8] = "Stabilized SmoothSCAD AUTHOR variant, not unrestricted published SURE"
    status[9] = "Hybrid SureShrink with sparse-level fallback"
    for i in (11, 23): status[i] = "PROVISIONAL 1D proxy, not a verified original published algorithm"
    status[15] = "1D orthonormal SURE-LET adaptation, not original image/interscale algorithm"
    status[19] = "Cai BlockJS formula, 1D periodic boundary convention"
    status[20] = "Jeffreys/ABE 1D shrinkage formula"
    status[22] = ("Chang-Yu-Vetterli (2000) CONTEXT, 1D orthonormal: 4 neighbors+parent, "
                  "LS context, L=max(50,2% level), adaptive soft threshold")
    status[24], status[25] = "Universal hard threshold", "Universal soft threshold"
    pd.DataFrame({"Index": np.arange(1, 27), "Procedure": E3_NAMES, "ValidationStatus": status}) \
        .to_csv(path, index=False)

def e3_write_summaries(results, cfg, out_dir):
    rows, paired, scores = [], [], []
    for st in results:
        mu, sd, Mr = st["mse"].mean(axis=0), st["mse"].std(axis=0, ddof=1), st["M"]
        scores.append(mu / mu[24])                            # relative to universal hard
        for m, nm in enumerate(E3_NAMES):
            rows.append([st["Signal"], st["SNR"], nm, Mr, mu[m], sd[m], sd[m] / np.sqrt(Mr)])
        for m in range(4, 24):
            for a_ in range(4):
                dif = st["mse"][:, m] - st["mse"][:, a_]
                dm, se = dif.mean(), dif.std(ddof=1) / np.sqrt(Mr)
                paired.append([st["Signal"], st["SNR"], E3_NAMES[a_], E3_NAMES[m], dm, se,
                               dm - 1.96 * se, dm + 1.96 * se, 100 * dm / mu[m]])
    T = pd.DataFrame(rows, columns=["Signal", "SNR", "Method", "M", "AMSE", "SD_MSE", "SE_AMSE"])
    T["N"] = cfg["N"]; T["RMSE"] = np.sqrt(T["AMSE"])
    T.to_csv(out_dir / "Experiment3_AMSE_all.csv", index=False)
    pd.DataFrame(paired, columns=["Signal", "SNR", "SCOPE", "Other", "OtherMinusSCOPE", "PairedSE",
                                  "CI_L", "CI_U", "ImprovementPct"]) \
        .to_csv(out_dir / "Experiment3_PairedDifferences.csv", index=False)
    Q = pd.DataFrame({"Method": E3_NAMES, "MeanRelativeAMSE_toHard": np.mean(scores, axis=0)})
    Q.sort_values("MeanRelativeAMSE_toHard", kind="stable").to_csv(out_dir / "Experiment3_AggregateRanking.csv", index=False)
    (out_dir / "Experiment3_README.txt").write_text(
        f"Independent comparison, M={cfg['M']} per signal-SNR; 4 SCOPE + 20 comparators + hard/soft.\n"
        f"Calibration: Section 4 rounded B_N={cfg['B_N']:g}, j0={cfg['j0']}; target B_N/2^j.\n"
        f"26 methods: #5 BAMS author Cscale=2 gamma=2 depth J-{cfg['j0']}; #23 Chang-Yu-Vetterli context 1D.\n"
        "Frozen F/k/r: Normal fixed r=0,1 k=5; Normal SCOPE SURE; Normal N-SCOPE SURE.\n"
        "Paired noisy realizations; root SNR=std(f,1)/std(eps), eps iid N(0,1).\n"
        "METHOD_IMPLEMENTATION_NOTES.csv identifies provisional 1D proxy methods.\n"
        "Python translation of SCOPE_Exp123_BN: NumPy MT19937 noise (not bit-identical to MATLAB randn).\n")

def run_experiment3(mode, N, Mrep, snrs, out_base, snr_tag, n_jobs=1, resume=True):
    smoke = mode == "smoke"
    log("Frozen E3: Normal fixed r=0, Normal fixed r=1, Normal SCOPE SURE, Normal N-SCOPE SURE.")
    B_N, j0 = scope_B_for_N(N)
    cfg = dict(N=N, J=int(round(np.log2(N))), j0=j0, B_N=B_N, M=Mrep, signals=list(SIGNALS), snrs=list(snrs),
               snrDefinition="std_signal_over_std_noise_iidN01", kGrid=[3, 5, 7], rGrid=[0, 1, 2, 3],
               bGrid=[0.65, 0.80, 1, 1.20, 1.50], seed=20271126 + N, shift=2,
               resume=resume, saveEvery=50)
    cfg["codeRevision"] = f"PY_E3_BNrounded_Normal_N{N}_M{Mrep}_B{B_N:.6g}_j0{j0}_FINAL26_AuthorBAMS_CYV_v1"
    if smoke:
        cfg.update(M=2, signals=["Doppler", "Blocks"], snrs=[5])
        out_dir = Path(out_base) / f"Exp3_N{N}_SMOKE"
    else:
        out_dir = Path(out_base) / f"Exp3_N{N}_M{Mrep}{snr_tag}"
    (out_dir / "plots").mkdir(parents=True, exist_ok=True)
    e3_write_audit(out_dir / "METHOD_IMPLEMENTATION_NOTES.csv")
    a = compute_constants(N, ["Normal"], cfg["rGrid"], cfg["kGrid"], cfg["j0"], cfg["B_N"],
                          cache=out_dir / "calibration_E3_Normal.pkl")
    check_constants(a, ["Normal"], cfg["rGrid"], cfg["kGrid"], N, cfg["B_N"])
    jobs = [(cfg, a, si + 1, iq + 1, s, snr, out_dir)
            for si, s in enumerate(cfg["signals"]) for iq, snr in enumerate(cfg["snrs"])]
    t0 = time.time()
    results = run_conditions(e3_condition_job, jobs, n_jobs)
    e3_write_summaries(results, cfg, out_dir)
    save_pickle(dict(cfg=cfg, results=results, a=a, names=E3_NAMES), out_dir / "Experiment3_Results.pkl")
    log(f"Experiment III finished: {out_dir}; {time.time() - t0:.1f} s")
    return out_dir

## Part 7 — Run the experiments

Mirrors the main loop of `SCOPE_Exp123_BN`: for each $N$, Experiments I,
II and III in turn, then the across-$N$ consolidation
(`AllN_Exp*_AMSE_RMSE*.csv`). Everything is resumable: re-running this cell
after an interruption continues from the last checkpoint.

Approximate single-core run times for the **full** design ($M=1000$, three
$N$, 12 conditions each): Experiment I a few minutes, Experiment II about
15–30 minutes, Experiment III about 1–1.5 hours (dominated by the 20
comparators at $N=2048$). Set `N_JOBS` in Part 0 to run conditions in parallel.

In [ ]:
NS_RUN, SNRS_RUN, M_RUN, SNR_TAG = resolve_design(MODE, NS, SNRS, M)
OUT = Path(OUT_BASE); OUT.mkdir(parents=True, exist_ok=True)
log(f"Design: mode={MODE}, experiments={EXPERIMENTS}, N={NS_RUN}, root SNR={SNRS_RUN}, M={M_RUN}")
log("Noise: N(0,1); root SNR = SD(signal)/1 (an SD ratio, not a power ratio).")

def consolidate_across_N(out_base, Ns, Mrep, experiments, snr_tag):
    """scope_consolidate_N: stack Experiment*_AMSE_all.csv over N (never pooled)."""
    for ie in (1, 2, 3):
        tag = f"Exp{ie}"
        if tag not in experiments:
            continue
        frames = [pd.read_csv(p) for N in Ns
                  if (p := Path(out_base) / f"{tag}_N{N}_M{Mrep}{snr_tag}" / f"Experiment{ie}_AMSE_all.csv").exists()]
        if frames:
            pd.concat(frames, ignore_index=True).to_csv(Path(out_base) / f"AllN_{tag}_AMSE_RMSE{snr_tag}.csv", index=False)

if RUN_SIMULATIONS:
    t_all = time.time()
    for N_ in NS_RUN:
        log(f"================ N = {N_} ================")
        if "Exp1" in EXPERIMENTS:
            run_experiment1(MODE, N_, M_RUN, SNRS_RUN, OUT, SNR_TAG, N_JOBS, RESUME)
        if "Exp2" in EXPERIMENTS:
            run_experiment2(MODE, N_, M_RUN, SNRS_RUN, OUT, SNR_TAG, N_JOBS, RESUME)
        if "Exp3" in EXPERIMENTS:
            run_experiment3(MODE, N_, M_RUN, SNRS_RUN, OUT, SNR_TAG, N_JOBS, RESUME)
    if MODE == "full":
        consolidate_across_N(OUT, NS_RUN, M_RUN, EXPERIMENTS, SNR_TAG)
    log(f"Completed. Results folder: {OUT.resolve()} ({time.time() - t_all:.1f} s)")

## Part 8 — Figures

All figures are rebuilt **from the CSV files written above**, so they can be
restyled and regenerated at any time without re-running simulations (set
`RUN_SIMULATIONS = False` in Part 0 to jump straight here).

### Figure style: fonts, sizes, LaTeX
Edit `PLOT_STYLE` and re-run the cells below.

* `use_tex`: `True` typesets every label with a real LaTeX installation
  (needs `latex`, `dvipng` and the `cm-super` fonts); `False` uses
  Matplotlib's built-in TeX-math renderer with Computer Modern fonts; `"auto"`
  tries LaTeX first and falls back automatically. All labels are written in
  a syntax that renders identically in both modes.
* Font sizes are in points; `fig_width` is in inches (6.5 ≈ a LaTeX text
  width, 3.3 ≈ one column).
* `formats` controls the saved files (vector PDF for the paper, PNG for
  quick viewing); `show` controls inline display (`"summary"` shows the
  cross-condition figures and saves the per-condition ones silently).

In [ ]:
import io, logging
import matplotlib as mpl
logging.getLogger("fontTools").setLevel(logging.ERROR)   # silence font-embedding chatter
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

PLOT_STYLE = dict(
    use_tex=False,             # False = fast mathtext; change to True or "auto" for full LaTeX
    font_family="serif",       # "serif" (Computer Modern look) or "sans-serif"
    base_size=10,              # default text
    title_size=11,             # axes titles
    label_size=10,             # axis labels
    tick_size=9,               # tick labels
    legend_size=8.5,           # legends
    method_size=8,             # method names on long categorical axes
    suptitle_size=12,          # figure titles
    fig_width=6.5,             # inches
    dpi=200,                   # PNG resolution
    formats=("pdf", "png"),    # saved formats
    show_titles=True,          # draw titles (turn off for camera-ready LaTeX captions)
    show="summary",            # "summary" | "all" | "none"
    grid_alpha=0.35,
)
FIG_DIR = Path(OUT_BASE) / "figures"

# Okabe-Ito colour-blind-safe palette.
OKABE = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#000000"]

def _tex_works():
    """Probe whether a full LaTeX pipeline is available to Matplotlib."""
    try:
        with mpl.rc_context({"text.usetex": True}):
            fig = plt.figure(figsize=(1, 1)); fig.text(0.5, 0.5, r"$\lambda_j$ (\%)")
            fig.savefig(io.BytesIO(), format="png"); plt.close(fig)
        return True
    except Exception:
        plt.close("all")
        return False

def apply_style(style=PLOT_STYLE):
    """Set Matplotlib rcParams from PLOT_STYLE; returns whether real LaTeX is used."""
    use_tex = style["use_tex"]
    if use_tex == "auto":
        use_tex = shutil.which("latex") is not None and _tex_works()
    serif = style["font_family"] == "serif"
    mpl.rcParams.update({
        "text.usetex": bool(use_tex),
        "font.family": style["font_family"],
        "font.serif": ["Computer Modern Roman", "CMU Serif", "cmr10", "DejaVu Serif"],
        "mathtext.fontset": "cm" if serif else "dejavusans",
        "axes.formatter.use_mathtext": True,
        "axes.unicode_minus": False,
        "font.size": style["base_size"],
        "axes.titlesize": style["title_size"],
        "axes.labelsize": style["label_size"],
        "xtick.labelsize": style["tick_size"],
        "ytick.labelsize": style["tick_size"],
        "legend.fontsize": style["legend_size"],
        "figure.titlesize": style["suptitle_size"],
        "axes.grid": True, "axes.axisbelow": True, "grid.alpha": style["grid_alpha"], "grid.linewidth": 0.6,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False, "savefig.bbox": "tight", "savefig.pad_inches": 0.03,
        "pdf.fonttype": 42, "ps.fonttype": 42,
        "axes.prop_cycle": mpl.cycler(color=OKABE),
    })
    if use_tex:
        mpl.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}\usepackage{amssymb}"
    log(f"Figure style: {'LaTeX (usetex)' if use_tex else 'Matplotlib mathtext, Computer Modern'}; "
        f"base font {style['base_size']} pt")
    return bool(use_tex)

USE_TEX = apply_style()

def save_fig(fig, name, subdir, summary=False):
    """Save in all requested formats; display inline according to PLOT_STYLE['show']."""
    d = FIG_DIR / subdir; d.mkdir(parents=True, exist_ok=True)
    for ext in PLOT_STYLE["formats"]:
        fig.savefig(d / f"{name}.{ext}", dpi=PLOT_STYLE["dpi"])
    show = PLOT_STYLE["show"]
    if show == "all" or (show == "summary" and summary):
        plt.show()
    plt.close(fig)

def title(ax, text):
    if PLOT_STYLE["show_titles"]:
        ax.set_title(text)

### LaTeX labels
Method, generator and axis labels used in every figure. They avoid characters
that would need different escaping in LaTeX and in Matplotlib's mathtext
(`%`, `_`, `&`), so the same strings work in both modes. Edit freely.

In [ ]:
PCT = r"$\%$"
L_AMSE = r"$\mathrm{AMSE}$"
L_SNR = r"root $\mathrm{SNR}=\mathrm{SD}(f)/\sigma$"

GEN_TEX = {"Normal": "Normal", "Logistic": "Logistic", "Laplace": "Laplace",
           "Sech": "Hyp. secant", "Uniform": "Uniform", "Cauchy": "Cauchy"}

E3_TEX = {
    "SCOPE_Fixed_Normal": r"Fixed SCOPE ($r=0$)",
    "NSCOPE_Fixed_Normal_r1": r"Fixed N-SCOPE ($r=1$)",
    "SCOPE_SURE_Normal": r"SCOPE SURE ($r=0$)",
    "NSCOPE_SURE_Normal": r"N-SCOPE SURE",
    "BAMS_Author_C2_Gamma2": r"BAMS (author, $C=2$, $\gamma=2$)",
    "EBayesMedian": r"EBayesThresh median",
    "RaisedCosineBayes": r"Raised-cosine Bayes",
    "EpanechnikovBayes": r"Epanechnikov Bayes",
    "SmoothSCAD_stabilized_author": r"Smooth SCAD (stabilized)",
    "SureShrink_Hybrid": r"SureShrink (hybrid)",
    "FDR_BH_q05": r"FDR ($q=0.05$)",
    "EBBlock_1D_proxy": r"EB block (1D proxy)",
    "BayesShrink_1D": r"BayesShrink",
    "NeighCoeff": r"NeighCoeff",
    "NeighShrink_SURE_1D": r"NeighShrink-SURE",
    "SURELET_1D_orthonormal": r"SURE-LET (1D)",
    "BiShrink_1D": r"BiShrink (1D)",
    "NeighBlock": r"NeighBlock",
    "SureBlock_hybrid_CZ09": r"SureBlock",
    "BlockJS": r"BlockJS",
    "Jeffreys_ABE_FigueiredoNowak": r"Jeffreys / ABE",
    "LocalWiener_1D": r"Local Wiener",
    "Context_CYV2000_1D": r"Context (CYV, 1D)",
    "HMT_1D_EM_proxy": r"HMT (1D EM proxy)",
    "UniversalHard": r"Universal hard",
    "UniversalSoft": r"Universal soft",
}
E2_METHOD_TEX = {"FixedSCOPE": r"Fixed SCOPE ($r=0$)", "FixedNSCOPE": r"Fixed N-SCOPE ($r=1$)",
                 "SCOPE_SURE": r"SCOPE SURE", "NSCOPE_SURE": r"N-SCOPE SURE"}

def snr_tex(s): return rf"$\mathrm{{SNR}}={fmt_snr(s)}$"
def cond_tex(signal, snr, Mrep, N=None):
    n = rf"$N={N}$, " if N is not None else ""
    return rf"{signal}, {n}{snr_tex(snr)}, $M={Mrep}$"

### Locating results
The figures use the result folders of the design in Part 0 (`MODE`, `NS`,
`SNRS`, `M`). Missing folders are skipped with a message.

In [ ]:
def result_dir(exp, N):
    if MODE == "smoke":
        return Path(OUT_BASE) / f"{exp}_N{N}_SMOKE"
    return Path(OUT_BASE) / f"{exp}_N{N}_M{M_RUN}{SNR_TAG}"

def available(exp):
    out = [(N_, result_dir(exp, N_)) for N_ in NS_RUN]
    found = [(N_, p) for N_, p in out if (p / f"Experiment{exp[-1]}_AMSE_all.csv").exists()]
    for N_, p in out:
        if (N_, p) not in found:
            log(f"  (no {exp} results for N={N_}: {p})")
    return found

def ordered(values, reference):
    """Keep the canonical order (signals, generators) for whatever is present."""
    return [v for v in reference if v in set(values)]

### Figure 0 — the four test signals with one noisy realisation
Signals scaled to the chosen root SNR (here the median of the design), with
one $N(0,1)$ noise draw; useful as Figure 1 of the paper.

In [ ]:
def fig_test_signals(N=None, snr=None, seed=1):
    N = N or NS_RUN[0]; snr = snr or SNRS_RUN[len(SNRS_RUN) // 2]
    rng = make_rng(seed); t = np.arange(1, N + 1) / N
    fig, axs = plt.subplots(2, 2, figsize=(PLOT_STYLE["fig_width"], 0.62 * PLOT_STYLE["fig_width"]),
                            sharex=True, constrained_layout=True)
    for ax, s in zip(axs.ravel(), SIGNALS):
        f = scale_to_snr(make_dj_signal(s, N)[0], snr)
        ax.plot(t, f + rng.standard_normal(N), lw=0.5, color="0.65", label="noisy $y$")
        ax.plot(t, f, lw=1.2, color=OKABE[0], label="signal $f$")
        title(ax, s); ax.set_xlim(0, 1)
    for ax in axs[1]: ax.set_xlabel(r"$t$")
    axs[1, 1].legend(loc="upper left")      # empty corner of the Blocks panel
    if PLOT_STYLE["show_titles"]:
        fig.suptitle(rf"Donoho-Johnstone test signals, $N={N}$, {snr_tex(snr)}, $\sigma=1$")
    save_fig(fig, f"F0_TestSignals_N{N}_SNR{fmt_snr(snr)}", "Exp0_signals", summary=True)

### Experiment I figures
* **Per condition** (`E1_AMSE_*`, `E1_GainPercent_*`): grouped bars of AMSE
  for the 6 generators × radii $r=0,1,2,3$, and the percentage AMSE reduction
  of $r=1,2,3$ relative to the same generator at $r=0$ (MATLAB
  `e1_make_barplots`).
* **Generator comparison** (paper figure `E1_gain_CDF`): reduction from
  $r=0$ to $r=1$ per generator and signal, averaged equally over root SNR.
* **Sample-size sensitivity** (MATLAB `scope_plot_N_SNR_sensitivity`,
  paper Figure 2): Normal $r=1$ vs $r=0$ against $N$, one curve per SNR.
  Needs at least two $N$ and two SNR values.

In [ ]:
def e1_condition_figures(T, N):
    for (s, snr), g in T.groupby(["Signal", "SNR"], sort=False):
        Mrep = int(g["M"].iloc[0])
        gens = ordered(g["Generator"].unique(), GENERATORS)
        bars = np.array([[g[(g.Generator == F) & (g.Radius == r)]["AMSE"].iloc[0] for r in range(4)] for F in gens])
        x = np.arange(len(gens)); w = 0.2
        fig, ax = plt.subplots(figsize=(PLOT_STYLE["fig_width"], 0.5 * PLOT_STYLE["fig_width"]))
        for r in range(4):
            ax.bar(x + (r - 1.5) * w, bars[:, r], w, label=rf"$r={r}$" + (" (SCOPE)" if r == 0 else ""))
        ax.set_xticks(x, [GEN_TEX[F] for F in gens]); ax.set_ylabel(L_AMSE)
        ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.0))
        ax.set_ylim(0, bars.max() * 1.18)
        title(ax, "Fixed SCOPE / N-SCOPE: " + cond_tex(s, snr, Mrep, N))
        save_fig(fig, f"E1_AMSE_{s}_SNR{fmt_snr(snr)}_N{N}", "Exp1")
        gains = 100 * (bars[:, [0]] - bars[:, 1:]) / bars[:, [0]]
        fig, ax = plt.subplots(figsize=(PLOT_STYLE["fig_width"], 0.5 * PLOT_STYLE["fig_width"]))
        for r in range(1, 4):
            ax.bar(x + (r - 2) * 0.26, gains[:, r - 1], 0.26, label=rf"$r={r}$", color=OKABE[r])
        ax.axhline(0, color="k", lw=0.8)
        ax.set_xticks(x, [GEN_TEX[F] for F in gens])
        ax.set_ylabel(rf"{L_AMSE} reduction vs. $r=0$ ({PCT})")
        ax.legend(ncol=3, loc="best")
        title(ax, "Neighbourhood gains: " + cond_tex(s, snr, Mrep, N))
        save_fig(fig, f"E1_GainPercent_{s}_SNR{fmt_snr(snr)}_N{N}", "Exp1")

def e1_gain_by_generator(T, N):
    """Paper figure E1_gain_CDF: r=0 -> r=1 reduction, averaged equally over SNR."""
    sigs = ordered(T["Signal"].unique(), SIGNALS); gens = ordered(T["Generator"].unique(), GENERATORS)
    G = np.zeros((len(gens), len(sigs)))
    for i, F in enumerate(gens):
        for k, s in enumerate(sigs):
            vals = []
            for snr, g in T[(T.Signal == s) & (T.Generator == F)].groupby("SNR"):
                a0 = g[g.Radius == 0]["AMSE"].iloc[0]; a1 = g[g.Radius == 1]["AMSE"].iloc[0]
                vals.append(100 * (a0 - a1) / a0)
            G[i, k] = np.mean(vals)
    x = np.arange(len(gens)); w = 0.8 / len(sigs)
    fig, ax = plt.subplots(figsize=(PLOT_STYLE["fig_width"], 0.48 * PLOT_STYLE["fig_width"]))
    for k, s in enumerate(sigs):
        ax.bar(x + (k - (len(sigs) - 1) / 2) * w, G[:, k], w, label=s)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(x, [GEN_TEX[F] for F in gens])
    ax.set_ylabel(rf"{L_AMSE} reduction, $r=0\to r=1$ ({PCT})")
    ax.legend(ncol=len(sigs), loc="lower center", bbox_to_anchor=(0.5, 1.0))
    snrs = sorted(T["SNR"].unique())
    if PLOT_STYLE["show_titles"]:
        ax.set_title(rf"Effect of $r=1$ by generator, $N={N}$, mean over $\mathrm{{SNR}}\in\{{{', '.join(fmt_snr(v) for v in snrs)}\}}$", pad=24)
    save_fig(fig, f"E1_gain_CDF_N{N}", "Exp1", summary=True)

def e1_sensitivity(frames):
    """Normal r=1 vs r=0 against N, one curve per SNR (paper Figure 2)."""
    T = pd.concat(frames, ignore_index=True)
    Ns, snrs = sorted(T["N"].unique()), sorted(T["SNR"].unique())
    if len(Ns) < 2 or len(snrs) < 2:
        log("  sensitivity figure needs >= 2 sample sizes and >= 2 SNR values; skipped")
        return
    sigs = ordered(T["Signal"].unique(), SIGNALS)
    fig, axs = plt.subplots(2, 2, figsize=(PLOT_STYLE["fig_width"], 0.78 * PLOT_STYLE["fig_width"]),
                            constrained_layout=True)
    markers = ["o", "s", "^", "D", "v"]
    for ax, s in zip(axs.ravel(), sigs):
        for k, snr in enumerate(snrs):
            gain = []
            for N_ in Ns:
                g = T[(T.N == N_) & (T.Signal == s) & (T.SNR == snr)]
                a0 = g[g.Procedure == "Normal_r0"]["AMSE"].iloc[0]; a1 = g[g.Procedure == "Normal_r1"]["AMSE"].iloc[0]
                gain.append(100 * (a0 - a1) / a0)
            ax.plot(Ns, gain, marker=markers[k % 5], lw=1.4, ms=5, label=snr_tex(snr))
        ax.axhline(0, color="k", lw=0.8, ls=":")
        ax.set_xscale("log", base=2); ax.set_xticks(Ns, [str(n) for n in Ns]); ax.minorticks_off()
        ax.set_xlabel(r"sample size $N$"); ax.set_ylabel(rf"{L_AMSE} reduction ({PCT})")
        title(ax, s)
    axs[0, 0].legend(loc="best")
    if PLOT_STYLE["show_titles"]:
        fig.suptitle(r"Fixed Normal N-SCOPE $r=1$ vs. SCOPE $r=0$")
    save_fig(fig, f"E1_Figure2_Sensitivity_N_SNR_M{M_RUN}{SNR_TAG}", "Exp1", summary=True)

def plot_experiment1():
    found = available("Exp1"); frames = []
    for N_, p in found:
        T = pd.read_csv(p / "Experiment1_AMSE_all.csv"); frames.append(T)
        e1_condition_figures(T, N_)
        e1_gain_by_generator(T, N_)
    if frames:
        e1_sensitivity(frames)

### Experiment II figures
* **Per condition** (`E2_AMSE_*`, `E2_Gains_*`, `E2_RadiusSelection_*`):
  AMSE of the four procedures per generator; their AMSE reduction relative
  to fixed SCOPE; and how often N-SCOPE SURE selects each radius.
* **Aggregate** (`E2_AggregateRelativeAMSE`): mean over all signal/SNR
  cases of AMSE / universal-hard AMSE.

In [ ]:
def e2_condition_figures(T, S, N):
    methods = list(E2_METHOD_TEX)
    for (s, snr), g in T[T.Generator != "Universal"].groupby(["Signal", "SNR"], sort=False):
        Mrep = int(g["M"].iloc[0]); gens = ordered(g["Generator"].unique(), GENERATORS)
        bars = np.array([[g[(g.Generator == F) & (g.Method == m)]["AMSE"].iloc[0] for m in methods] for F in gens])
        x = np.arange(len(gens)); w = 0.2
        fig, ax = plt.subplots(figsize=(PLOT_STYLE["fig_width"], 0.5 * PLOT_STYLE["fig_width"]))
        for i, m in enumerate(methods):
            ax.bar(x + (i - 1.5) * w, bars[:, i], w, label=E2_METHOD_TEX[m])
        ax.set_xticks(x, [GEN_TEX[F] for F in gens]); ax.set_ylabel(L_AMSE)
        ax.set_ylim(0, bars.max() * 1.25); ax.legend(ncol=2, loc="upper center")
        title(ax, "Fixed and SURE SCOPE / N-SCOPE: " + cond_tex(s, snr, Mrep, N))
        save_fig(fig, f"E2_AMSE_{s}_SNR{fmt_snr(snr)}_N{N}", "Exp2")
        gains = 100 * (bars[:, [0]] - bars[:, 1:]) / bars[:, [0]]
        fig, ax = plt.subplots(figsize=(PLOT_STYLE["fig_width"], 0.5 * PLOT_STYLE["fig_width"]))
        for i in range(3):
            ax.bar(x + (i - 1) * 0.26, gains[:, i], 0.26, label=E2_METHOD_TEX[methods[i + 1]], color=OKABE[i + 1])
        ax.axhline(0, color="k", lw=0.8)
        ax.set_xticks(x, [GEN_TEX[F] for F in gens])
        ax.set_ylabel(rf"{L_AMSE} reduction vs. fixed SCOPE ({PCT})"); ax.legend(ncol=3, loc="best")
        title(ax, "Adaptive and neighbourhood gains: " + cond_tex(s, snr, Mrep, N))
        save_fig(fig, f"E2_Gains_{s}_SNR{fmt_snr(snr)}_N{N}", "Exp2")
        sel = S[(S.Signal == s) & (S.SNR == snr) & (S.Method == "NSCOPE_SURE") & (S.Parameter == "r")]
        radii = sorted(sel["Value"].unique())
        freq = np.array([[sel[(sel.Generator == F) & (sel.Value == r)]["Frequency"].iloc[0] for r in radii] for F in gens])
        fig, ax = plt.subplots(figsize=(PLOT_STYLE["fig_width"], 0.5 * PLOT_STYLE["fig_width"]))
        bottom = np.zeros(len(gens))
        for i, r in enumerate(radii):
            ax.bar(x, 100 * freq[:, i], 0.6, bottom=bottom, label=rf"$r={int(r)}$"); bottom += 100 * freq[:, i]
        ax.set_xticks(x, [GEN_TEX[F] for F in gens]); ax.set_ylim(0, 100)
        ax.set_ylabel(rf"selected radius ({PCT} of replications)")
        ax.legend(ncol=len(radii), loc="upper center", bbox_to_anchor=(0.5, 1.13))
        if PLOT_STYLE["show_titles"]:
            ax.set_title("N-SCOPE SURE radius selection: " + cond_tex(s, snr, Mrep, N), pad=22)
        save_fig(fig, f"E2_RadiusSelection_{s}_SNR{fmt_snr(snr)}_N{N}", "Exp2")

def e2_aggregate(R, N):
    gens = ordered([p.split("_")[0] for p in R["Procedure"] if not p.startswith("Universal")], GENERATORS)
    methods = list(E2_METHOD_TEX)
    vals = np.array([[R[R.Procedure == f"{F}_{m}"]["MeanRelativeAMSE_to_UniversalHard"].iloc[0]
                      for m in methods] for F in gens])
    x = np.arange(len(gens)); w = 0.2
    fig, ax = plt.subplots(figsize=(PLOT_STYLE["fig_width"], 0.5 * PLOT_STYLE["fig_width"]))
    for i, m in enumerate(methods):
        ax.bar(x + (i - 1.5) * w, vals[:, i], w, label=E2_METHOD_TEX[m])
    ax.axhline(1, color="k", lw=0.8, ls=":")
    ax.set_xticks(x, [GEN_TEX[F] for F in gens])
    ax.set_ylabel(r"mean $\mathrm{AMSE}/\mathrm{AMSE}_{\mathrm{hard}}$")
    ax.set_ylim(0, vals.max() * 1.25); ax.legend(ncol=2, loc="upper center")
    title(ax, rf"Experiment II: aggregate non-oracle performance, $N={N}$")
    save_fig(fig, f"E2_AggregateRelativeAMSE_N{N}", "Exp2", summary=True)

def plot_experiment2():
    for N_, p in available("Exp2"):
        T = pd.read_csv(p / "Experiment2_AMSE_all.csv")
        S = pd.read_csv(p / "Experiment2_SelectionFrequencies.csv")
        R = pd.read_csv(p / "Experiment2_AggregateRanking.csv")
        e2_condition_figures(T, S, N_)
        e2_aggregate(R, N_)

### Experiment III figures
* **Per condition** (`E3_AMSE_*`): AMSE of the 24 non-reference methods,
  sorted, SCOPE recipes highlighted; dashed / dotted lines mark universal
  hard / soft thresholding.
* **Per condition** (`E3_MSEBoxes_*`): replication-MSE distributions
  (5–95% whiskers, quartile box, median) of the four SCOPE recipes and the
  six best comparators — paired, since all methods share the noise.
* **Aggregate ranking** (`E3_AggregateRanking`): mean over the 12 conditions
  of AMSE / universal-hard AMSE.
* **Signal profiles** (paper Figure `E3_signal_profile`): the same ratio per
  signal, averaged equally over root SNR; methods sorted by overall mean.
* **Relative-risk heat map** (supplementary): AMSE / hard for every method
  and condition.

In [ ]:
SCOPE_COLOR, COMP_COLOR, PROXY_COLOR = OKABE[0], "0.62", OKABE[1]
PROXIES = {"EBBlock_1D_proxy", "HMT_1D_EM_proxy"}

def method_axis(ax, labels, axis="y"):
    (ax.set_yticks if axis == "y" else ax.set_xticks)(range(len(labels)), labels, fontsize=PLOT_STYLE["method_size"])

def e3_condition_figures(T, p, N):
    for (s, snr), g in T.groupby(["Signal", "SNR"], sort=False):
        Mrep = int(g["M"].iloc[0])
        mu = g.set_index("Method")["AMSE"].reindex(E3_NAMES)
        comp = E3_NAMES[:24]
        order = sorted(comp, key=lambda n: mu[n])
        fig, ax = plt.subplots(figsize=(PLOT_STYLE["fig_width"], 0.95 * PLOT_STYLE["fig_width"]))
        ax.barh(range(24), [mu[n] for n in order], color=method_colors_sorted(order), height=0.72)
        ax.axvline(mu["UniversalHard"], color="k", ls="--", lw=1.1, label=E3_TEX["UniversalHard"])
        ax.axvline(mu["UniversalSoft"], color="k", ls=":", lw=1.1, label=E3_TEX["UniversalSoft"])
        method_axis(ax, [E3_TEX[n] for n in order]); ax.invert_yaxis()
        ax.set_xlim(0, 1.08 * max(mu.max(), 1e-12)); ax.set_xlabel(L_AMSE); ax.grid(axis="y", visible=False)
        ax.legend(loc="lower right")
        title(ax, cond_tex(s, snr, Mrep, N))
        save_fig(fig, f"E3_AMSE_{s}_SNR{fmt_snr(snr)}_N{N}", "Exp3")
        rep_file = p / f"MSE_{s}_SNR{fmt_snr(snr)}.csv"
        if rep_file.exists():
            R = pd.read_csv(rep_file)
            best6 = sorted(E3_NAMES[4:24], key=lambda n: mu[n])[:6]
            ids = E3_NAMES[:4] + best6
            fig, ax = plt.subplots(figsize=(PLOT_STYLE["fig_width"], 0.55 * PLOT_STYLE["fig_width"]))
            for i, n in enumerate(ids):
                q = np.quantile(R[n].to_numpy(), [0.05, 0.25, 0.5, 0.75, 0.95])
                col = SCOPE_COLOR if i < 4 else COMP_COLOR
                ax.plot([i, i], [q[0], q[4]], color="0.3", lw=0.9)
                ax.add_patch(plt.Rectangle((i - 0.29, q[1]), 0.58, max(q[3] - q[1], 1e-12),
                                           facecolor=col, alpha=0.45, edgecolor="0.2"))
                ax.plot([i - 0.29, i + 0.29], [q[2], q[2]], color="k", lw=1.4)
            ax.axhline(mu["UniversalHard"], color="k", ls="--", lw=1, label=E3_TEX["UniversalHard"])
            ax.axhline(mu["UniversalSoft"], color="k", ls=":", lw=1, label=E3_TEX["UniversalSoft"])
            ax.set_xticks(range(len(ids)), [E3_TEX[n] for n in ids], rotation=35, ha="right",
                          fontsize=PLOT_STYLE["method_size"])
            ax.set_xlim(-0.6, len(ids) - 0.4); ax.set_ylabel(r"replication $\mathrm{MSE}$")
            ax.legend(loc="best")
            title(ax, cond_tex(s, snr, Mrep, N) + ": SCOPE and the six best comparators")
            save_fig(fig, f"E3_MSEBoxes_{s}_SNR{fmt_snr(snr)}_N{N}", "Exp3")

def method_colors_sorted(order):
    return [SCOPE_COLOR if n in E3_NAMES[:4] else (PROXY_COLOR if n in PROXIES else COMP_COLOR) for n in order]

def e3_aggregate(Q, N):
    Q = Q[Q.Method.isin(E3_NAMES[:24])].sort_values("MeanRelativeAMSE_toHard")
    fig, ax = plt.subplots(figsize=(PLOT_STYLE["fig_width"], 0.95 * PLOT_STYLE["fig_width"]))
    ax.barh(range(len(Q)), Q["MeanRelativeAMSE_toHard"], color=method_colors_sorted(list(Q.Method)), height=0.72)
    ax.axvline(1, color="k", ls="--", lw=1)
    method_axis(ax, [E3_TEX[n] for n in Q.Method]); ax.invert_yaxis(); ax.grid(axis="y", visible=False)
    ax.set_xlabel(r"mean $\mathrm{AMSE}/\mathrm{AMSE}_{\mathrm{hard}}$ over all conditions")
    title(ax, rf"Experiment III: aggregate relative performance, $N={N}$")
    save_fig(fig, f"E3_AggregateRanking_N{N}", "Exp3", summary=True)

def e3_relative_matrix(T):
    """Ratio AMSE / AMSE_hard for every method (rows) and condition (columns)."""
    piv = T.pivot_table(index="Method", columns=["Signal", "SNR"], values="AMSE")
    sigs = ordered(T["Signal"].unique(), SIGNALS)
    cols = [(s, v) for s in sigs for v in sorted(T["SNR"].unique()) if (s, v) in piv.columns]
    piv = piv[cols]
    return piv.div(piv.loc["UniversalHard"], axis=1)

def e3_signal_profile(T, N):
    rel = e3_relative_matrix(T).loc[E3_NAMES[:24]]
    sigs = ordered(T["Signal"].unique(), SIGNALS)
    prof = pd.DataFrame({s: rel[[c for c in rel.columns if c[0] == s]].mean(axis=1) for s in sigs})
    order = prof.mean(axis=1).sort_values().index
    fig, axs = plt.subplots(1, len(sigs), figsize=(PLOT_STYLE["fig_width"], 1.0 * PLOT_STYLE["fig_width"]),
                            sharey=True, constrained_layout=True)
    axs = np.atleast_1d(axs)
    for ax, s in zip(axs, sigs):
        v = prof.loc[order, s]
        ax.hlines(range(len(order)), 0, v, color="0.8", lw=0.8)
        ax.scatter(v, range(len(order)), c=method_colors_sorted(list(order)), s=22, zorder=3)
        ax.axvline(1, color="k", ls="--", lw=0.9)
        ax.set_xlim(0, max(1.05, v.max() * 1.08)); ax.grid(axis="y", visible=False)
        title(ax, s); ax.set_xlabel(r"$\mathrm{AMSE}/\mathrm{AMSE}_{\mathrm{hard}}$")
    method_axis(axs[0], [E3_TEX[n] for n in order]); axs[0].invert_yaxis()
    if PLOT_STYLE["show_titles"]:
        fig.suptitle(rf"Experiment III signal profiles, $N={N}$ (mean over root SNR)")
    save_fig(fig, f"E3_signal_profile_Normal_N{N}", "Exp3", summary=True)

def e3_heatmap(T, N):
    rel = e3_relative_matrix(T).loc[E3_NAMES]
    fig, ax = plt.subplots(figsize=(PLOT_STYLE["fig_width"], 1.05 * PLOT_STYLE["fig_width"]))
    vals = np.log2(rel.to_numpy())
    lim = np.nanmax(np.abs(vals)) if np.isfinite(vals).any() else 1
    im = ax.imshow(vals, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
    for (i, j), v in np.ndenumerate(rel.to_numpy()):
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=PLOT_STYLE["method_size"] - 1.5,
                color="w" if abs(np.log2(v)) > 0.6 * lim else "k")
    method_axis(ax, [E3_TEX[n] for n in rel.index])
    ax.set_xticks(range(rel.shape[1]), [rf"{s}, {fmt_snr(v)}" for s, v in rel.columns],
                  rotation=60, ha="right", fontsize=PLOT_STYLE["method_size"])
    ax.grid(False)
    cb = fig.colorbar(im, ax=ax, shrink=0.6, pad=0.02)
    cb.set_label(r"$\log_2(\mathrm{AMSE}/\mathrm{AMSE}_{\mathrm{hard}})$")
    title(ax, rf"Experiment III relative risk, $N={N}$ (signal, root SNR)")
    save_fig(fig, f"E3_RelativeRiskHeatmap_N{N}", "Exp3", summary=True)

def plot_experiment3():
    for N_, p in available("Exp3"):
        T = pd.read_csv(p / "Experiment3_AMSE_all.csv")
        Q = pd.read_csv(p / "Experiment3_AggregateRanking.csv")
        e3_condition_figures(T, p, N_)
        e3_aggregate(Q, N_)
        e3_signal_profile(T, N_)
        e3_heatmap(T, N_)

### Draw everything
Re-run this cell after changing `PLOT_STYLE` (and re-running the style cell).
Files go to `SCOPE_Exp123_results_BNrounded/figures/<experiment>/`.

In [ ]:
apply_style()
fig_test_signals()
plot_experiment1()
plot_experiment2()
plot_experiment3()
n_files = sum(1 for _ in FIG_DIR.rglob("*.pdf")) if FIG_DIR.exists() else 0
log(f"Figures written to {FIG_DIR.resolve()} ({n_files} PDF files)")